# nb03 · Gauge-X so với HM trên VGG16: Bảng 1 mở rộng (566 ảnh test)

**Mục tiêu.** So sánh phương pháp đề xuất **Gauge-X** với phương pháp của Hidden Monotonicity (HM). Hai bên dùng đúng các ảnh, các hàng Split của Bảng 1 và 6 thước đo Quantus như nb01. Notebook làm ba việc:
1. tính và chấm điểm hai họ mới, **HM với đạo hàm đúng** và **Gauge-X**, trên 566 ảnh test của ImageNet-S-50;
2. đặt chúng cạnh 9 hàng mà nb01 đã chấm bằng code HM (6 hàng Split, LayerCAM tầng 14 và 2, Random), thành một Bảng 1 mở rộng;
3. so sánh ghép cặp theo ảnh và kết luận theo các giả thuyết ở §3.7.

**HM không được chạy lại.** Điểm của HM (code chính thức) lấy nguyên từ nb01. nb03 chỉ tính bản đồ `hm` bằng engine, để đối chiếu với bản đồ nb01 đã lưu ở từng ảnh (kiểm cầu nối) và để so bản đồ ở §11; các bản đồ này không được chấm điểm.

**Ba họ phương pháp.** Cả ba dùng chung công thức bản đồ của HM; chúng chỉ khác nhau ở các núm điều chỉnh gauge.

| Họ | Thay cho α | Thay cho θ, Θ | Lượt về Conv2d | Nguồn điểm |
| --- | --- | --- | --- | --- |
| `hm`: code chính thức | α = 0.4 | θ = 0.1, Θ = 10 | đổi chỗ, như code | nb01 |
| `hm_true`: HM với đạo hàm đúng | α = 0.4 | θ = 0.1, Θ = 10 | đúng | nb03 |
| `gauge`: **Gauge-X**, phương pháp đề xuất | $\alpha^\star_l$ theo quy tắc cân bằng | cân bằng $\lVert\sigma\rVert = \lVert a\rVert$ | đúng | nb03 |

**Các hàng.** 6 hàng Split của Bảng 1, giữ nguyên tầng, kiểu maxpool và hậu xử lý của HM. Không có bước chọn cấu hình nào, nên Gauge-X không dùng ảnh nào để tinh chỉnh. SplitLRP không dùng α và cũng không dùng lượt về Conv2d, nên ở hàng 6 họ `hm_true` trùng `hm` và không được tính lại. Tổng cộng có 11 biến thể được chấm cho mỗi ảnh.

**Cách chạy.** Chỉ cần đổi `N_IMAGES` ở §1:
- `1`: chạy thử trên CPU. Mọi cell chạy, nhưng Pixel Flipping và Selectivity dùng ít bước hơn, nên số liệu chưa có giá trị kết luận.
- `566`: chạy đủ, trên mọi GPU nhìn thấy được, mỗi GPU `WORKERS_PER_GPU` worker.

Từ terminal, trong thư mục `gauge-centering-xai/experiments`:
```
uv run python scripts/run_notebook.py nb03                                                 # chạy thử: 1 ảnh, CPU
uv run python scripts/run_notebook.py nb03 --n-images 566 --gpus all --workers-per-gpu 3   # chạy đủ
uv run python scripts/run_notebook.py nb03 --n-images 566 --gpus 0,1 --workers-per-gpu 4   # chỉ GPU 0 và 1
```
Kết quả nằm ở `results/nb03/n{N_IMAGES}/`. Nếu lần chạy bị ngắt, chạy lại đúng lệnh đó: các ảnh đã xong được giữ nguyên, chỉ các ảnh còn lại được tính.

**Cần có trước khi chạy:**
- lần chạy nb01 với cùng số ảnh hoặc nhiều hơn (chạy đủ: `results/nb01/n566/`), gồm `data/`, `configs/` và `seed42/fp32/` (bản đồ, ảnh đầu vào và mask của từng ảnh);
- ảnh trong `docs/data/` (`scripts/prepare_data.py`);
- repo HM chính thức ở commit `31be19b`.

**Lưu kết quả, tách tính toán và hiển thị.** Như các notebook trước, mỗi mục có hai loại cell:
- **cell tính toán**: tính rồi lưu số liệu gốc vào `data/` (CSV, NPZ; tên cột tiếng Anh cố định như `method`, `row`, `family`, `metric`);
- **cell hiển thị**: dòng đầu là `# ▶ Hiển thị`; cell này chỉ đọc `data/` rồi vẽ hình hoặc lập bảng qua `scripts/nb03_report.py`.

Tên hiển thị lấy từ `LABELS` (§15). Muốn đổi nhãn thì sửa `LABELS` rồi chạy lại các cell hiển thị, không cần tính lại.

**Code dùng.**
- `scripts/`: `gaugexai.py` (engine), `nb03_worker.py` (toàn bộ việc tính trên một ảnh, chạy song song), `nb03_report.py`, `nb_progress.py`, `quantus_batched.py`; thêm `dcpair.py` của nb02, chỉ cho phép kiểm ở §4.
- Kết quả và config của nb01: điểm baseline, tham số các phương pháp HM và các thước đo.
- Repo HM chính thức: bộ nạp dữ liệu, `QuantusEvaluator` (điểm số được tính bằng đúng code của nb01) và phép kiểm khớp ở §4.

## 1. Cấu hình

Chỉ cần đổi `N_IMAGES`. Mọi tham số khác đều có lý do ở bảng bên dưới. Cell này chỉ đặt hằng số và đường dẫn, không tính gì.

**Thiết bị và song song.**
- `GPUS` là danh sách GPU cho các worker. Mặc định: chạy thử dùng CPU; chạy đủ dùng mọi GPU mà `nvidia-smi` thấy. `run_notebook.py --gpus` đổi được giá trị này (`all`, `cpu` hoặc danh sách như `0,1,2,3`), qua biến `NB_GPUS`.
- Mỗi GPU chạy `WORKERS_PER_GPU` worker (`--workers-per-gpu`, biến `NB_WORKERS_PER_GPU`).
- Kernel của notebook dùng GPU đầu tiên cho các phép kiểm ở §4–§5.
- TF32 bị tắt, như nb01, để float32 là float32 thật.

In [ ]:
import os, sys
from pathlib import Path
N_IMAGES = 1          # <- số duy nhất cần đổi: 1 = chạy thử (CPU), 566 = chạy đủ (toàn bộ tập test, mọi GPU)

N_TEST = 566
assert 1 <= N_IMAGES <= N_TEST
SMOKE = N_IMAGES < N_TEST
EXP_DIR = Path.cwd().resolve()
assert (EXP_DIR / "pyproject.toml").exists(), "hãy mở notebook từ thư mục gauge-centering-xai/experiments"
sys.path.insert(0, str(EXP_DIR / "scripts"))
from nb03_worker import available_cpus, gpu_inventory, nb01_mask, attrloc_undefined   # light helpers: no torch

_gpus = os.environ.get("NB_GPUS", "cpu" if SMOKE else "all").strip().lower()
_inventory = gpu_inventory()                       # before this kernel touches a GPU: shows other jobs only
GPUS = ([g["index"] for g in _inventory] if _gpus == "all" else
        [] if _gpus in ("cpu", "", "-1") else [int(s) for s in _gpus.split(",")])
GPU_INFO = [g for g in _inventory if g["index"] in GPUS]
WORKERS_PER_GPU = int(os.environ.get("NB_WORKERS_PER_GPU", 3))
assert GPUS or SMOKE, "chạy đủ cần GPU: không thấy GPU nào (nvidia-smi), hoặc đặt --gpus"
N_WORKERS = min(N_IMAGES, len(GPUS) * WORKERS_PER_GPU if GPUS else 1)
CPU_THREADS = max(1, available_cpus() // N_WORKERS)
os.environ["CUDA_VISIBLE_DEVICES"] = str(GPUS[0]) if GPUS else ""   # the kernel's device; must precede `import torch`
import json, time, gc, copy, platform, subprocess, logging, importlib, warnings, contextlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torchvision
import torchvision.transforms as T
from IPython.display import display, Markdown

SPLIT_SEED = 42
T_GRID = np.linspace(0.0, 1.0, 41)
H1_BAND = (0.33, 0.42)
QUANTUS_CHUNK = int(os.environ.get("NB_QUANTUS_CHUNK", 64))
TF32 = False
N_FIG = min(2, N_IMAGES)
N_SCORE_CHECK = min(3, N_IMAGES)
CPU_METRIC_PARAMS = {"pixel_flipping": {"features_in_step": 1568}, "selectivity": {"patch_size": 28}}
BRIDGE_TOL = {"rel_diff": 1e-4, "off_pixels": 1e-3, "one_minus_spearman": 1e-4}
SCORE_TOL = 1e-3
HM_COMMIT = "31be19b460358a69516452402d41bd650845a344"

# the six Split rows of HM's Table 1: id, Table-1 row, method, layer (input of that chunk, HM's convention), map kind,
# max-pool form, HM's post-processing, HM config file and formula (fidelity check of §4)
ROWS = pd.DataFrame([
    ("r1", 1, "SplitCAM", "features_12", "cam_g", "convex", "none", "splitcam_a40_sc.json", "f12_cam_g"),
    ("r2", 2, "SplitCAM", "features_14", "cam_g", "convex", "none", "splitcam_a40_sc.json", "f14_cam_g"),
    ("r3", 3, "SplitCAM", "features_26", "cam_pp", "wta", "abs", "splitcam_a40_sc_wta_f26.json", "f26_cam_pp_abs"),
    ("r4", 4, "SplitGrad", "features_12", "grad", "wta", "none", "splitgrad_a40_wta.json", "f12_grad_pp"),
    ("r5", 5, "SplitGrad", "features_14", "grad", "wta", "abs", "splitgrad_a40_wta.json", "f14_grad_pp_abs"),
    ("r6", 6, "SplitLRP", "features_26", "lrp", "wta", "abs", "splitlrp_sc_wta.json", "f26_cam_pp_abs"),
], columns=["id", "table1_row", "method", "layer", "kind", "pool", "hm_post", "hm_config", "hm_formula"])
MAP_FAMILIES = ["hm", "hm_true", "gauge"]       # computed by the engine; 'hm' only for the bridge and §11
NEW_FAMILIES = ["hm_true", "gauge"]             # scored by nb03
VARIANTS = [(r.id, f, r.hm_post) for r in ROWS.itertuples() for f in NEW_FAMILIES
            if not (r.kind == "lrp" and f == "hm_true")]
PAIRS = ["hm~hm_true", "gauge~hm", "gauge~hm_true"]


def nb01_run():
    """Smallest nb01 run that covers the first N_IMAGES test images (the full run needs results/nb01/n566)."""
    runs = sorted((int(d.name[1:]), d) for d in (EXP_DIR / "results" / "nb01").glob("n*")
                  if d.name[1:].isdigit() and (d / "data" / "long_seed42.csv").exists())
    runs = [(n, d) for n, d in runs if n >= N_IMAGES and (SMOKE or n == N_TEST)]
    return runs[0][1] if runs else None

NB01_RUN = Path(os.environ["NB01_RUN"]).resolve() if "NB01_RUN" in os.environ else nb01_run()
assert NB01_RUN is not None, (f"chưa có lần chạy nb01 nào phủ {N_IMAGES} ảnh test đầu (cần results/nb01/n"
                              f"{N_TEST if not SMOKE else '…'}/data/long_seed42.csv): chạy nb01 trước")

PARAMS = pd.DataFrame([
    ("N_IMAGES", N_IMAGES, "1 = chạy thử; 566 = toàn bộ tập test của HM, đúng các ảnh nb01 đã chấm"),
    ("baseline", str(NB01_RUN.relative_to(EXP_DIR)) if NB01_RUN.is_relative_to(EXP_DIR) else str(NB01_RUN),
     "điểm của 9 hàng Bảng 1 do nb01 tính bằng code HM; nb03 không chạy lại HM"),
    ("các hàng", "6 hàng Split của Bảng 1", "giữ tầng, maxpool, hậu xử lý của HM; không chọn lại cấu hình, nên Gauge-X "
                                            "không dùng ảnh nào để tinh chỉnh"),
    ("biến thể chấm điểm", len(VARIANTS), "hm_true ở hàng 1–5 và gauge ở hàng 1–6; SplitLRP không dùng α và lượt về "
                                          "conv, nên hm_true ở hàng 6 trùng HM"),
    ("thước đo", "6 thước đo, config của lần chạy nb01", "bản chụp configs/metrics_4 và metrics_maxsens của nb01, chấm bằng "
                                                         "QuantusEvaluator của HM: cùng code và cùng tham số với baseline"),
    ("ảnh đầu vào, mask, lớp", "của nb01", "baseline_original_image.npz, baseline_binary_mask.npz và lớp dự đoán trong "
                                           "metadata.json của nb01: hai bên giải thích cùng một tensor và cùng một lớp"),
    ("dtype", "float32 (TF32 tắt)", "như nb01. Trong toạ độ (d, σ) không có phép trừ hai số lớn, nên float32 đủ; §4 (v) "
                                    "kiểm bằng số. Trên RTX 4090, đỉnh lý thuyết của TF32 (tensor core, dày đặc) bằng FP32 "
                                    "(82.6 TFLOPS), nên tắt TF32 ở đây ít thiệt hơn trên A6000 (TF32 gấp đôi FP32)"),
    ("α, θ, Θ của HM", "0.4, 0.1, 10", "configs/nb01/methods/*.json (các hàng Split của Bảng 1)"),
    ("T_GRID", "41 điểm trên [0, 1]", "bước α = 0.0125, mịn gấp 4 lần lưới α của HM (bước 0.05); chứa t = 0.2 (α = 0.4); "
                                     "lưới chỉ để tìm khoảng chứa nghiệm, rồi regula falsi 4 vòng trên log tỉ số (§3.5)"),
    ("H1_BAND", H1_BAND, "dải α mà HM chọn qua tinh chỉnh (0.35–0.4, Bảng 8–9 của HM) nới thêm nửa bước lưới α của HM "
                         "mỗi bên, làm tròn"),
    ("Max Sens", "seed 42·10⁴ + k", "mọi biến thể của ảnh k bắt đầu từ cùng seed, nên hm_true và Gauge-X gặp cùng các ảnh "
                                    "nhiễu. nb01 cũng đặt seed này cho ảnh k nhưng chấm các hàng HM nối tiếp trên cùng dòng số "
                                    "ngẫu nhiên, nên nhiễu của HM khác: hiệu với HM ghép cặp theo ảnh, không theo nhiễu"),
    ("QUANTUS_CHUNK", QUANTUS_CHUNK, "như nb01: Pixel Flipping và Selectivity đưa 64 bước nhiễu qua VGG16 mỗi lần "
                                     "(scripts/quantus_batched.py, chỉ GPU); cùng kích thước lô nên làm tròn như baseline. "
                                     "Biến NB_QUANTUS_CHUNK giảm được số này để chạy thử trên GPU nhỏ"),
    ("GPUS", GPUS if GPUS else "CPU", "chạy thử: CPU; chạy đủ: mọi GPU nvidia-smi thấy (đổi bằng --gpus)"),
    ("WORKERS_PER_GPU", WORKERS_PER_GPU, "mỗi worker làm trọn một ảnh. Vòng tạo ảnh nhiễu của Quantus và phần engine nhỏ "
                                         "chạy trên CPU hoặc chờ đồng bộ, nên nhiều worker trên một GPU giữ GPU luôn bận; "
                                         "đỉnh bộ nhớ của từng worker được đo ở §13"),
    ("N_WORKERS", N_WORKERS, "số GPU × WORKERS_PER_GPU (không quá số ảnh); ảnh được chia động, GPU nhanh làm nhiều ảnh hơn"),
    ("CPU_THREADS", CPU_THREADS, "số CPU khả dụng (theo cgroup của container) chia đều cho các worker"),
    ("tham số thước đo khi chạy thử trên CPU", json.dumps(CPU_METRIC_PARAMS) if SMOKE and not GPUS else "như nb01",
     "không có GPU thì không có quantus_batched: 96 bước Pixel Flipping, 64 ô Selectivity thay cho 1344 và 784 "
     "(Quantus đòi features_in_step chia hết 224·224)"),
    ("BRIDGE_TOL", json.dumps(BRIDGE_TOL), "bản đồ hm của engine so với bản đồ nb01 lưu: sai lệch L2 tương đối như §4 (ii) "
                                          "float32; SplitLRP đo bằng tỉ lệ pixel và Spearman như §4 (v) (quy tắc argmax "
                                          "của LRP gián đoạn ở điểm gần hoà)"),
    ("SCORE_TOL", SCORE_TOL, "§5: điểm LayerCAM chấm lại so với nb01, sai lệch tương đối; cuDNN làm tròn khác nhau giữa "
                             "các GPU ở mức 10⁻⁶, còn sai đường chấm (ảnh, mask, lớp, tham số) cho sai lệch lớn hơn nhiều"),
    ("N_FIG, N_SCORE_CHECK", f"{N_FIG}, {N_SCORE_CHECK}", "số ảnh minh hoạ (§12); số ảnh của phép kiểm đường chấm (§5)"),
    ("HM_COMMIT", HM_COMMIT[:7], "phiên bản repo HM của nb01"),
], columns=["param", "value", "reason"])

NB_ID = "nb03"
DOCS = EXP_DIR.parents[1]
HM_REPO = Path(os.environ.get("HM_REPO", DOCS / "DC-Decomposition-for-Explanability")).resolve()
DATA_ROOT = Path(os.environ.get("DATA_ROOT", DOCS / "data")).resolve()
IMAGENET, IMAGENET_S = DATA_ROOT / "imagenet", DATA_ROOT / "imagenet-s"
NB01_MAIN = NB01_RUN / f"seed{SPLIT_SEED}" / "fp32"                # nb01's per-image folders (maps, input, mask)
RUN_DIR = EXP_DIR / "results" / NB_ID / f"n{N_IMAGES}"
DATA_DIR = RUN_DIR / "data"                                         # raw numbers behind every table and figure
SCORE_DIR, PER_IMAGE_DIR, MAP_DIR = DATA_DIR / "scores", DATA_DIR / "per_image", DATA_DIR / "maps"
for d in (DATA_DIR, SCORE_DIR, PER_IMAGE_DIR, MAP_DIR, RUN_DIR / "figures", RUN_DIR / "tables", RUN_DIR / "configs"):
    d.mkdir(parents=True, exist_ok=True)

import gaugexai as gx                              # (d, sigma) engine: the families, balanced rule, SplitLRP
import nb03_report as rep                          # tables and figures, read only from DATA_DIR
from nb_progress import Progress                   # progress bar + progress.json for scripts/run_notebook.py
LABELS = rep.default_labels()                      # display names; edit them in §15
H2_CELLS = len(ROWS) * len(rep.METRIC_KEYS)        # cells of H2: 6 Split rows x 6 metrics

DEVICE = torch.device("cuda" if torch.cuda.device_count() > 0 else "cpu")   # is_available() stays True on Windows with no visible GPU
torch.backends.cudnn.allow_tf32 = torch.backends.cuda.matmul.allow_tf32 = TF32
if DEVICE.type == "cpu":
    torch.set_num_threads(available_cpus())
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 120)
PARAMS.to_csv(DATA_DIR / "params.csv", index=False)
ROWS.to_csv(DATA_DIR / "rows.csv", index=False)
print(f"N_IMAGES = {N_IMAGES} ({'CHẠY THỬ' if SMOKE else 'CHẠY ĐỦ'}); worker: {N_WORKERS} trên "
      f"{'GPU ' + ','.join(map(str, GPUS)) if GPUS else 'CPU'}; {len(VARIANTS)} biến thể chấm điểm")
print(f"baseline: {NB01_RUN}\ndữ liệu : {DATA_ROOT}\nkết quả : {RUN_DIR}")

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
display(rep.show_csv(RUN_DIR, "params.csv", LABELS))
display(rep.show_csv(RUN_DIR, "rows.csv", LABELS))

## 2. Môi trường, dữ liệu, baseline nb01

Cell tính toán làm các việc sau:
- kiểm repo HM đúng commit `31be19b` và không có thay đổi cục bộ;
- nạp 616 ảnh ImageNet-S-50 bằng chính bộ nạp của HM và lấy 566 ảnh test theo đúng thứ tự của nb01;
- kiểm baseline nb01 cho `N_IMAGES` ảnh đầu:
  - thư mục ảnh của nb01 có ảnh đầu vào, mask, `metadata.json` và bản đồ của 6 hàng Split;
  - `long_seed42.csv` có đủ 9 hàng × 6 thước đo cho mọi ảnh;
  - ô NaN chỉ được phép ở hai chỗ: Max Sens của Random (không có trong Bảng 1), và Attr. Loc. khi Quantus trả NaN theo định nghĩa (tỉ lệ attribution trong mask lớn hơn 1, xem §3.7). Mỗi ô Attr. Loc. NaN được kiểm lại trên bản đồ nb01 đã lưu;
  - có bản chụp config thước đo của lần chạy nb01;
- nạp VGG16 (`IMAGENET1K_V1`), dựng engine, ghi danh sách GPU (`nvidia-smi`).

Lớp được giải thích ở ảnh k là lớp dự đoán mà nb01 ghi trong `metadata.json`, nên ba họ giải thích cùng một lớp.

In [ ]:
def git(*args):
    return subprocess.run(["git", "-C", str(HM_REPO), *args], capture_output=True, text=True, check=True).stdout.strip()

hm_commit, hm_dirty = git("rev-parse", "HEAD"), git("status", "--porcelain", "--untracked-files=no")
assert hm_commit == HM_COMMIT, f"repo HM đang ở {hm_commit}, cần {HM_COMMIT}"
assert hm_dirty == "", f"repo HM có thay đổi cục bộ:\n{hm_dirty}"
assert (IMAGENET / "val").is_dir() and (IMAGENET_S / "ImageNetS50" / "validation-segmentation").is_dir(), \
    f"thiếu ảnh trong {DATA_ROOT}: chạy scripts/prepare_data.py"
warnings.filterwarnings("ignore", category=SyntaxWarning)       # Quantus 0.6 docstrings
warnings.filterwarnings("ignore", message="IProgress not found")
sys.path.insert(0, str(HM_REPO))

@contextlib.contextmanager
def hm_quiet():
    """HM logs every tensor at DEBUG level and prints progress: keep both out of the notebook."""
    logging.disable(logging.CRITICAL)
    try:
        with open(RUN_DIR / "console.log", "a", encoding="utf-8") as fh, contextlib.redirect_stdout(fh), \
                contextlib.redirect_stderr(fh):
            yield
    finally:
        logging.disable(logging.NOTSET)
        root = logging.getLogger()                # importing HM sets the root logger to DEBUG
        root.setLevel(logging.WARNING)
        for h in root.handlers:
            h.setLevel(logging.WARNING)

with hm_quiet():
    import quantus
    from xai.evaluation.dataset_utils import _load_dataset
    from xai.evaluation.quantus_evaluator import QuantusEvaluator
    from xai.evaluation.map_generation import _initialize_hybrid_module, _compute_explanation_with_module
    import dcpair as dc
    import quantus_batched
    LOG = logging.getLogger("nb03")
    dataset, test_idx = _load_dataset(True, str(IMAGENET), str(IMAGENET_S), "ImageNetS50", -1, SPLIT_SEED, False, LOG)
    val_idx = [int(i) for i in dataset.get_stratified_indices(n_samples=50, random_seed=SPLIT_SEED)]
test_idx = [int(i) for i in test_idx]

# nb01 baseline: per-image folders, Table-1 row -> HM method, scores, metric configs
R2M = pd.read_csv(NB01_RUN / "data" / "row_to_method.csv").set_index("row")
NB01_LONG = pd.read_csv(NB01_RUN / "data" / "long_seed42.csv")
NB01_DIRS = {int(d.name.split("_")[1]): d for d in NB01_MAIN.glob("image_*") if d.is_dir()}
IMAGES, problems = [], []
for k, i in enumerate(test_idx[:N_IMAGES]):
    name = dataset.samples[i]["image_name"]
    rec = dict(image=k, dataset_index=i, image_name=name, class_id=dataset.samples[i]["class_id"])
    d = NB01_DIRS.get(k)
    if d is None or not d.name.endswith(name):
        problems.append(f"ảnh {k}: không có thư mục nb01 cho {name}")
    else:
        rec.update(target=int(json.loads((d / "metadata.json").read_text())["pred_class"]),
                   nb01_input=str(d / "baseline_original_image.npz"), nb01_mask=str(d / "baseline_binary_mask.npz"),
                   nb01_maps={r.id: str(d / f"{R2M.loc[r.table1_row, 'result_method']}.npz") for r in ROWS.itertuples()},
                   nb01_layercam14=str(d / f"{R2M.loc[7, 'result_method']}.npz"))
        need = [rec["nb01_input"], rec["nb01_mask"], rec["nb01_layercam14"], *rec["nb01_maps"].values()]
        problems += [f"ảnh {k}: thiếu {Path(f).name}" for f in need if not Path(f).exists()]
    IMAGES.append(rec)
names = [r["image_name"] for r in IMAGES]
mine = NB01_LONG[NB01_LONG["image_name"].isin(names)]
required = [(n, row, m) for n in names for row in range(1, 10) for m in rep.METRIC_KEYS]
missing = set(required) - set(zip(mine["image_name"], mine["row"], mine["metric"]))
unexplained_nan = []                              # NaN that is neither Random's Max Sens nor Quantus' undefined Attr. Loc.
for r in mine[mine["value"].isna()].itertuples():
    if r.row == 9 and r.metric == "max_sensitivity":
        continue
    d = NB01_DIRS.get(names.index(r.image_name))
    if r.metric == "attribution_localisation" and d is not None and attrloc_undefined(
            np.load(d / f"{R2M.loc[r.row, 'result_method']}.npz")["explanation"],
            nb01_mask(np.load(d / "baseline_binary_mask.npz")["explanation"])):
        continue
    unexplained_nan.append((r.image_name, r.row, r.metric))
NB01_METRIC_FILES = sorted([*(NB01_RUN / "configs" / "metrics_4").glob("*.json"),
                            *(NB01_RUN / "configs" / "metrics_maxsens").glob("*.json")])
checks = pd.Series({
    "616 ảnh nạp được (mask khớp nhãn ImageNet)": len(dataset) == 616,
    "test: 566 ảnh, rời tập val": len(test_idx) == N_TEST and set(val_idx).isdisjoint(test_idx),
    f"nb01: thư mục, ảnh đầu vào, mask, bản đồ cho {N_IMAGES} ảnh đầu": not problems,
    f"nb01: đủ điểm 9 hàng × 6 thước đo ({len(required)} ô)": not missing,
    "nb01: ô NaN chỉ là Max Sens của Random hoặc Attr. Loc. không xác định (tỉ lệ > 1)": not unexplained_nan,
    "nb01: config 5 thước đo": len(NB01_METRIC_FILES) == 5,
})
checks.rename_axis("check").rename("passed").reset_index().to_csv(DATA_DIR / "data_checks.csv", index=False)
assert checks.all(), f"{checks[~checks]}\n{problems[:5]}\nô thiếu: {sorted(missing)[:5]}\nNaN lạ: {unexplained_nan[:5]}"
pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("nb01_")} for r in IMAGES]
             ).to_csv(DATA_DIR / "image_split.csv", index=False)

TF = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(),
                T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])   # HM's load_image

def load_test(k):
    """Test image k as nb01 evaluated it: input tensor (1, 3, 224, 224), the mask nb01's evaluator got, class, PIL image."""
    rec, raw = IMAGES[k], dataset[IMAGES[k]["dataset_index"]][0]
    x = torch.from_numpy(np.load(rec["nb01_input"])["explanation"]).unsqueeze(0).float()
    return x, nb01_mask(np.load(rec["nb01_mask"])["explanation"]), rec["target"], raw

torchvision.models.VGG16_Weights.IMAGENET1K_V1.get_state_dict(progress=False)          # into the torch cache once
MODEL = torchvision.models.vgg16(weights="IMAGENET1K_V1").eval().to(DEVICE)
NET = gx.SplitNet(MODEL)
ROW_DICTS = ROWS[["id", "layer", "kind", "pool", "hm_post"]].to_dict("records")
if DEVICE.type == "cuda":
    quantus_batched.install(QUANTUS_CHUNK)        # §5 scores nb01's LayerCAM maps with the same batching as nb01
ENV = {"python": platform.python_version(), "platform": platform.platform(), "torch": torch.__version__,
       "torchvision": torchvision.__version__, "cuda": torch.version.cuda, "numpy": np.__version__,
       "pandas": pd.__version__, "quantus": quantus.__version__, "kernel_device": str(DEVICE),
       "gpus": [f"{g['index']}: {g['name']}, {g['memory_total_mib'] / 1024:.0f} GiB, đang dùng "
                f"{g['memory_used_mib'] / 1024:.1f} GiB, {g['utilization_pct']}%" for g in GPU_INFO],
       "cpus_available": available_cpus(), "n_workers": N_WORKERS, "cpu_threads_per_worker": CPU_THREADS,
       "hm_commit": hm_commit, "nb01_run": str(NB01_RUN), "n_images": N_IMAGES, "smoke": SMOKE, "tf32": TF32}
(DATA_DIR / "env.json").write_text(json.dumps(ENV, indent=2, ensure_ascii=False), encoding="utf-8")
busy = [g for g in GPU_INFO if g["utilization_pct"] > 10 or g["memory_used_mib"] > 2048]
print(f"môi trường, dữ liệu, baseline nb01: đạt; {N_IMAGES} ảnh test; thiết bị kernel {DEVICE}")
if busy:
    print("CHÚ Ý: GPU đang có việc khác, thời gian sẽ dài hơn:", [g["index"] for g in busy])

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
display(rep.show_env(RUN_DIR, LABELS))
display(rep.show_csv(RUN_DIR, "data_checks.csv", LABELS))
display(pd.read_csv(DATA_DIR / "image_split.csv").head(min(N_IMAGES, 10)))

## 3. Lý thuyết

**Ký hiệu.**
- Chunk $k$ của mạng tách là một tầng của VGG16. Tên chunk theo nút torch.fx của code HM: `x`, `features_0`, …, `avgpool`, `flatten`, `classifier_0`, …, `classifier_6`.
- Trạng thái $k$ là **đầu vào** của chunk $k$; đây cũng là quy ước tầng của code HM (`features_12` là đầu vào của conv thứ 13).
- $a_k$ là kích hoạt của mạng gốc tại trạng thái $k$, $f$ là logit của lớp được giải thích, $D_k = \partial f/\partial a_k$.
- HM mang hai luồng $(p_k, m_k)$. Đặt
$$d_k = p_k - m_k, \qquad \sigma_k = p_k + m_k .$$

### 3.1 Lượt đi: $d$ là mạng gốc, $\sigma$ là một truy hồi không âm

**Mệnh đề 1.** Với cấu hình của code chính thức (sau mỗi chunk: co theo ngưỡng, rồi ghi đè theo kích hoạt cache), ở mọi trạng thái $k \ge 1$:
1. $d_k = a_k$;
2. $\sigma$ được tính tiếp theo loại chunk như bảng dưới, rồi $\sigma \leftarrow \theta\sigma$ nếu $\max(\lvert p\rvert, \lvert m\rvert) > \Theta$, trong đó $\max(\lvert p\rvert, \lvert m\rvert) = \max \tfrac12(\lvert\sigma\rvert + \lvert a\rvert)$:

| Chunk | $\sigma_{k+1}$ |
| --- | --- |
| Conv2d, Linear $(W, b)$ | $\lvert W\rvert\,\sigma_k + \lvert b\rvert$ |
| ReLU (tiền kích hoạt $z = a_k$) | $\sigma_k + \mathrm{relu}(-z)$ |
| MaxPool, dạng lồi | $\mathrm{maxpool}(a_k) + \Sigma_{\text{cửa sổ}}(\sigma_k - a_k)$ |
| MaxPool, wta | $\sigma_k$ lấy tại argmax của $a_k$ trong mỗi cửa sổ |
| AdaptiveAvgPool | $\mathrm{avgpool}(\sigma_k)$ |
| Flatten, Dropout | $\sigma_k$ |

3. Tầng đọc ảnh nhận cặp $(x, 0)$, nên $\sigma = x$ tại đó. Chunk `x` xuất $(x/2, -x/2)$, nhưng không tầng nào đọc đầu ra này (`hybrid_network.py`).

*Chứng minh.*
(1) Bước ghi đè đặt $e = a - (p - m)$, rồi $p \leftarrow p + e/2$, $m \leftarrow m - e/2$. Sau bước này $p - m = a$, còn $p + m$ không đổi.

(2) Tính $p' + m'$ theo từng quy tắc tách của HM:
- affine: $p' = W^+p + W^-m + b^+$ và $m' = W^-p + W^+m + b^-$, nên $p' + m' = (W^+ + W^-)(p + m) + b^+ + b^- = \lvert W\rvert\sigma + \lvert b\rvert$;
- ReLU (maxout): $(p', m') = (\max(p, m), m)$. Nếu $z = p - m > 0$ thì $p' + m' = p + m$; nếu không thì $p' + m' = 2m = \sigma - z$. Gộp hai trường hợp được $\sigma + \mathrm{relu}(-z)$;
- maxpool lồi: $p' = \max_j(p_j - m_j) + \sum_j m_j$ và $m' = \sum_j m_j$. Vì $2m = \sigma - d$, ta có $p' + m' = \max_j d_j + \sum_j(\sigma_j - d_j)$;
- maxpool wta: $(p', m') = (p_{j^\star}, m_{j^\star})$ với $j^\star = \arg\max_j(p_j - m_j) = \arg\max_j a_j$;
- co: nhân cả hai luồng với $\theta$ thì $\sigma$ nhân $\theta$; bước ghi đè ngay sau đó khôi phục $d$.

Cuối cùng, $\max(\lvert p\rvert, \lvert m\rvert) = \tfrac12\max(\lvert\sigma + d\rvert, \lvert\sigma - d\rvert) = \tfrac12(\lvert\sigma\rvert + \lvert d\rvert)$. $\square$

**Hệ quả.** Hai luồng của HM là $\big(\tfrac{\sigma_k + a_k}{2}, \tfrac{\sigma_k - a_k}{2}\big)$; θ và Θ chỉ tác động lên σ.

### 3.2 Lượt về: phần cổ điển cộng phần gauge

HM khởi đầu bốn độ nhạy $(pp, np, pn, nn) = (e/2, 0, 0, e/2)$ ở đầu ra, với $e$ là véc-tơ đơn vị của lớp đích. Qua mỗi chunk, từ cuối về đầu, code làm ba bước:
1. lan truyền ngược với mẫu ReLU/argmax của mạng gốc;
2. dịch α: $pp, np \mathrel{-}= \alpha(pp + np)$ và $pn, nn \mathrel{-}= \alpha(pn + nn)$;
3. ghi đè theo gradient cache.

**Mệnh đề 2.** Đặt $t = 1 - 2\alpha$. Với đạo hàm đúng ở conv, tại đầu vào chunk $l$:
$$pp = \tfrac14(D + \Gamma + S),\quad np = \tfrac14(-D - \Gamma + S),\quad pn = \tfrac14(-D + \Gamma + S),\quad nn = \tfrac14(D - \Gamma + S).$$
Ở đây $(\Gamma, S)$ là **truy hồi gauge**: khởi đầu $(\Gamma, S) = (0, e)$ ở đầu ra; qua mỗi chunk, từ cuối về đầu,
$$\Gamma \leftarrow B_{dd}\,\Gamma + B_{d\sigma}\,S, \qquad S \leftarrow B_{\sigma\sigma}\,S, \qquad S \leftarrow t\,S,$$
với $B_{dd}$ là lan truyền ngược của mạng gốc, còn $B_{\sigma\sigma}$ và phần "tiêm" $B_{d\sigma}$ cho bởi bảng:

| Chunk | $B_{\sigma\sigma}$ | $B_{d\sigma}$ |
| --- | --- | --- |
| Conv2d, Linear | $\lvert W\rvert^\top$ | 0 |
| ReLU | 1 | $-(1 - \text{mask})$ |
| MaxPool lồi | $\Sigma^\top$ | scatter về argmax $-\ \Sigma^\top$ |
| MaxPool wta | scatter về argmax | 0 |
| AvgPool, Flatten, Dropout | như $B_{dd}$ | 0 |

Hệ quả: $\delta^g := pp - np = \tfrac12(D + \Gamma)$, $\delta^h := pn - nn = \tfrac12(\Gamma - D)$, và $\delta^g - \delta^h = D$.

*Chứng minh.* Đổi sang độ nhạy theo $(d, \sigma)$: với đầu ra $g$, $G_d = \tfrac12(pp - np)$ và $G_\sigma = \tfrac12(pp + np)$; tương tự $H_d, H_\sigma$ cho $h$.

Lượt về qua một chunk là chuyển vị của phép tuyến tính hoá lượt đi (Mệnh đề 1) tại $x^\star$:
- affine: $d' = Wd + b$, $\sigma' = \lvert W\rvert\sigma + \lvert b\rvert$, nên $(G_d, G_\sigma) \leftarrow (W^\top G_d,\ \lvert W\rvert^\top G_\sigma)$;
- ReLU: $d' = \mathrm{relu}(d)$, $\sigma' = \sigma + \mathrm{relu}(-d)$, nên $\partial d'/\partial d = \text{mask}$, $\partial\sigma'/\partial d = -(1 - \text{mask})$, $\partial\sigma'/\partial\sigma = 1$;
- maxpool lồi: $\partial\sigma'/\partial d$ có chuyển vị là scatter $-\ \Sigma^\top$, và $\partial\sigma'/\partial\sigma$ có chuyển vị là $\Sigma^\top$;
- maxpool wta: cả hai luồng đều định tuyến về argmax.

Code HM dùng đúng các mẫu của mạng gốc.

Dịch α: vì $pp + np = 2G_\sigma$, ta có $G_\sigma \leftarrow (1 - 2\alpha)G_\sigma$, còn $G_d$ không đổi.

Bước ghi đè cộng cùng một lượng $c$ vào $pp, nn$ và trừ $c$ khỏi $np, pn$. Kết quả: $G_d \mathrel{+}= c$, $H_d \mathrel{-}= c$, còn $G_\sigma$, $H_\sigma$ và $G_d + H_d$ không đổi. Bước này chỉ ép $G_d - H_d = D/2$.

Cuối cùng, tách điểm khởi đầu theo tính tuyến tính: $(G_d, G_\sigma) = (\tfrac14e, 0) + (0, \tfrac14e)$ và $(H_d, H_\sigma) = (-\tfrac14e, 0) + (0, \tfrac14e)$.
- Với hạt giống $(\pm\tfrac14e, 0)$, độ nhạy theo $\sigma$ luôn bằng 0, còn độ nhạy theo $d$ đi qua $B_{dd}$, cho ra $\pm\tfrac14D$.
- Hạt giống $(0, \tfrac14e)$, chung cho $g$ và $h$, cho ra $(\tfrac14\Gamma, \tfrac14S)$.

Cộng hai phần lại được công thức cần chứng minh. $\square$

### 3.3 Mỗi núm của HM tác động vào đâu

Các bản đồ được tính tại đầu vào chunk $l$, rồi phóng lên 224×224 bằng nearest như code HM:

| Bản đồ của code HM | Công thức | θ, Θ | α |
| --- | --- | --- | --- |
| `cam_g` $= \sum_c (pp - np)(p - m)$ | $\tfrac12\sum_c(D + \Gamma)\,a$ | không | có |
| `grad_only` $= \mathrm{mean}_c(pp - np)$ (SplitGrad) | $\tfrac12\,\mathrm{mean}_c(D + \Gamma)$ | không | có |
| `cam_pp` $= \sum_c pp \cdot p$ | $\tfrac18\sum_c(D + \Gamma + S)(\sigma + a)$ | có | có |
| `lrp_combined` $= \mathrm{mean}_c(R^+ - R^-)$ | quy tắc LRP của code trên $\big(\tfrac{\sigma + a}{2}, \tfrac{\sigma - a}{2}\big)$ | có | không |

Ghi chú:
- Hàng SplitGrad của Bảng 1 ghi `grad_pp`, nhưng code bỏ qua khoá này và tính `grad_only` (`map_generation.py`). Tương tự, hàng SplitLRP ghi `cam_pp`, còn code tính `lrp_combined`.
- Quy tắc LRP của code khác bài báo:
  - hai luồng được chia cho tổng chuẩn L1 của cả hai, rồi kẹp dưới ở $10^{-5}$;
  - không dùng bias; tỉ số không hữu hạn được đặt bằng 0;
  - maxpool định tuyến theo argmax của chính từng luồng.

  Engine làm đúng như code để bản đồ khớp.
- Phần cổ điển $D$ là gradient thường. Mọi thông tin thêm của các phương pháp Split nằm trong phần gauge: $\Gamma$, $S$ và $\sigma$.

### 3.4 Lỗi đổi chỗ ở conv trong toạ độ $(d, \sigma)$

Lượt về Conv2d của code chính thức trả $np$ và $pn$ đổi chỗ cho nhau (`direct_conv2d_split.py`; Linear thì đúng). Đặt
$$c = \tfrac12(G_d - H_d), \quad \gamma = \tfrac12(G_d + H_d), \quad s = \tfrac12(G_\sigma + H_\sigma), \quad \Delta = \tfrac12(G_\sigma - H_\sigma).$$

**Mệnh đề 3.** Phép đổi chỗ $np \leftrightarrow pn$ đúng bằng phép đổi $\gamma \leftrightarrow \Delta$, còn $c$ và $s$ giữ nguyên.

*Chứng minh.* Sau khi đổi chỗ:
- $G_d' = \tfrac12(pp - pn) = \tfrac12(G_d + G_\sigma - H_d - H_\sigma) = c + \Delta$;
- $H_d' = \tfrac12(np - nn) = \tfrac12(-G_d + G_\sigma + H_d - H_\sigma) = -c + \Delta$.

Vậy $c' = c$ và $\gamma' = \Delta$. Tương tự $G_\sigma' = \gamma + s$ và $H_\sigma' = -\gamma + s$, nên $s' = s$ và $\Delta' = \gamma$. $\square$

Hệ quả:
- **Với đạo hàm đúng, $\Delta \equiv 0$.** Trong code chính thức, $\Delta$ đi qua $B_{\sigma\sigma}$ và bị nhân $t$ như $S$. Nó chỉ chảy vào $c$, phần mà bước ghi đè xoá đi.
- **Tại đầu vào một conv**, phần gauge của $\delta^g$ trong code chính thức là giá trị của $\Delta$ ngay trước conv đó, tức một bản sao qua $\lvert W\rvert^\top$ của phần gauge ở conv phía trên. Ở conv cuối (`features_28`), $\Delta = 0$, nên phần gauge bị xoá hẳn.
- **Engine tái hiện code chính thức bằng một dòng**: đổi $\Gamma \leftrightarrow \Delta$ sau mỗi Conv2d. SplitLRP không dùng lượt về này, nên không bị ảnh hưởng.

### 3.5 Các họ và quy tắc cân bằng

| Họ | $t = 1 - 2\alpha$ | σ | Conv2d |
| --- | --- | --- | --- |
| `hm` | 0.2 | co θ = 0.1 khi vượt Θ = 10 | đổi chỗ, như code |
| `hm_true` | 0.2 | như `hm` | đúng |
| `gauge` (Gauge-X) | $t^\star_l$ theo quy tắc cân bằng | cân bằng $\lVert\hat\sigma_k\rVert = \lVert a_k\rVert$ sau mỗi chunk | đúng |
| `g0` (chỉ dùng trong phép kiểm (iv) của §4) | 0 | 0 | — |

**Quy tắc cân bằng.** $t^\star_l = \min\{t \in [0, 1] : \lVert\Gamma_l(t)\rVert = \lVert D_l\rVert\}$ và $\alpha^\star_l = (1 - t^\star_l)/2$. Nếu không có $t$ nào đạt, lấy $t^\star_l = 1$, tức $\alpha^\star_l = 0$.

**Mệnh đề 4.** $\Gamma_l(0) = 0$, và $\Gamma_l(t)$ là đa thức theo $t$. Vì vậy, nếu $\max_{t \le 1}\lVert\Gamma_l(t)\rVert \ge \lVert D_l\rVert$ thì $t^\star_l$ tồn tại và xác định duy nhất.

*Chứng minh.* $\Gamma$ khởi đầu bằng 0 và chỉ nhận đóng góp từ $S$ qua $B_{d\sigma}$. Mỗi lần qua một chunk, $S$ bị nhân $t$, nên mỗi đóng góp là một đơn thức theo $t$. Với $t = 0$, $S$ bằng 0 ngay sau chunk cuối, nên $\Gamma \equiv 0$.

Hàm $t \mapsto \lVert\Gamma_l(t)\rVert$ liên tục, bằng 0 tại $t = 0$ và đạt giá trị $\ge \lVert D_l\rVert$ ở đâu đó trong $[0, 1]$. Theo định lý giá trị trung gian, tập nghiệm của $\lVert\Gamma_l(t)\rVert = \lVert D_l\rVert$ khác rỗng; tập này đóng, nên có phần tử nhỏ nhất, chính là $t^\star_l$. $\square$

**Tính số.** Một lượt về với lưới 41 giá trị $t$, gộp thành một batch, cho đường tỉ số $\lVert\Gamma_l(t)\rVert / \lVert D_l\rVert$ và khoảng đầu tiên chứa nghiệm. Đường này tăng gần như hàm mũ theo $t$, vì nó là tổng các đơn thức bậc cao. Vì vậy ta tinh chỉnh nghiệm bằng regula falsi trên log tỉ số: 4 vòng, mỗi vòng một lượt về với một hàng cho mỗi tầng. §4 kiểm rằng tỉ số tại $t^\star$ bằng 1.

**Vì sao α\* gần như không đổi theo ảnh (giải thích heuristic).** Đóng góp vào $\Gamma_l$ từ một điểm tiêm cách $n$ chunk có dạng $t^n$ nhân với tích các $\lvert W\rvert^\top$. Nếu mỗi chunk khuếch đại chuẩn khoảng $\rho$ lần, đóng góp đó cỡ $(t\rho)^n$: nhỏ dần khi $t < 1/\rho$ và nổ khi $t > 1/\rho$. Do đó tỉ số nhảy nhiều bậc độ lớn quanh $t_c \approx 1/\rho$, và $\alpha^\star \approx (1 - t_c)/2$ do trọng số quyết định chứ không do ảnh. Nếu $\alpha = 0.4$ của HM nằm sát điểm chuyển này, lần tinh chỉnh của HM thực chất là đi tìm $t_c$. §10 đo điều này.

**Gauge-X** là đúng công thức của HM, chỉ thay hai thứ:
- α thay bằng $\alpha^\star_l$ (bậc 1 của gauge);
- θ, Θ thay bằng phép cân bằng σ (bậc 0).

Mọi thứ khác giữ nguyên, kể cả đầu vào $(x, 0)$ và quy tắc LRP của code. $S$ đi cùng $t^\star_l$, không thêm hệ số nào khác. Engine tính $f$ bằng mạng gốc, nên không có phép trừ hai số lớn: chạy được float32, không cần cache hay θ. Quy tắc cân bằng được giải lại trên mọi ảnh, kể cả các ảnh nhiễu của Max Sens: đó là định nghĩa của phương pháp, không phải một tham số chọn trước.

*Ghi chú so với kế hoạch đã duyệt:* kế hoạch có thêm hệ số $\lVert\sigma_l\rVert/\lVert a_l\rVert$ cho $S$ trong `cam_pp`. Khi σ đã được cân bằng sau từng chunk thì không còn một "$\sigma_l$ chưa chuẩn hoá" xác định, nên hệ số đó trở thành một núm thứ ba. Ta bỏ nó: θ và Θ được thay đúng chỗ của chúng, giống cách code HM đối xử với θ.

**G0** ($t = 0$, $\sigma = 0$) cho đúng các bản đồ cổ điển: `cam_g` $= \tfrac12$ Grad×Act, SplitGrad $= \tfrac12$ Saliency, `cam_pp` $= \tfrac18$ Grad×Act. Ở đây G0 không được chấm điểm; nó chỉ dùng để kiểm phần $D$ của engine bằng autograd (§4 (iv)).

### 3.6 C và các bậc của gauge

Khai triển một gauge lồi $q$ tại $x^\star$ thành ba phần:
- bậc 0: $q(x^\star)$;
- bậc 1: $\langle\nabla q(x^\star), x - x^\star\rangle$;
- bậc 2: độ cong, nơi tham số $C$ của Paper1 nằm.

Bản đồ là gradient tại $x^\star$, nên chỉ phụ thuộc bậc 0 (qua giá trị luồng, ở `cam_pp` và LRP) và bậc 1 (qua $\Gamma$ và $S$). Bậc 2 không bao giờ đổi bản đồ tại $x^\star$.

Gauge-X chỉnh bậc 0 (cân bằng σ) và bậc 1 ($t^\star$), nên **không có $C$ để chọn**.

Ở bậc 1, mọi cách chọn $t$ đều cho gradient của một cặp DC hợp lệ, vì trừ cùng một hàm tuyến tính khỏi hai thành phần lồi vẫn giữ tính lồi (cái có thể mất là tính đơn điệu). Phát biểu này đúng nhưng không mang thông tin. Cái phân biệt các họ là **hướng** gauge $\Gamma, S$ mà HM dựng và **lượng** gauge giữ lại.

$C$ chỉ cần khi phải dựng gauge mới: mạng không có cặp đệ quy sẵn, bảo đảm trên cả vùng, hoặc DCA. Khi đó quy tắc là lấy cận trên chặt chẽ nhỏ nhất của $\mu(\Omega) = \max\big(0, -\min_\Omega \lambda_{\min}\nabla^2 f\big)$.

### 3.7 Giả thuyết và cách chấm

Mọi so sánh đều **ghép cặp theo ảnh**: với mỗi ảnh $k$ lấy hiệu $d_k$ giữa hai phương pháp, rồi báo $\bar d \pm \mathrm{SE}$ với $\mathrm{SE} = s_d/\sqrt n$. Một ô được đánh dấu theo hướng tốt của thước đo:
- `+`: tốt hơn quá 2·SE;
- `−`: kém hơn quá 2·SE;
- `=`: trong 2·SE.

Các giả thuyết:
- **H2 (chính).** Gauge-X không kém HM (code chính thức): ở 6 hàng Split × 6 thước đo (36 ô) của hiệu Gauge-X − HM, không ô nào là `−`. ĐẠT nếu không có ô `−`. Kèm số ô `+`, `=`, `−`.
  - *Đa so sánh.* Quy tắc này xét 36 ô cùng lúc. Nếu Gauge-X và HM thực sự bằng nhau, mỗi ô vẫn có khoảng 2.3% khả năng ra `−` do ngẫu nhiên; với 36 ô (không độc lập), khả năng có ít nhất một ô `−` không nhỏ. Vì vậy §14 ghi thêm số ô kém vượt ngưỡng Bonferroni cho 36 ô ($z < -3.20$, mức một phía 0.025/36), để người đọc phân biệt một ô `−` đơn lẻ với một xu hướng kém thật.
- **H1.** Ở mỗi tổ hợp (tầng, maxpool) mà các hàng của HM dùng, $\alpha^\star$ trung bình (đạo hàm đúng) nằm trong $[0.33, 0.42]$. Năm tổ hợp: (12, lồi), (14, lồi), (12, wta), (14, wta), (26, wta). Đạt nếu cả năm đều đạt.
- **H4.** Chỉ đo, không dự đoán trước: lỗi đổi chỗ ở conv làm bản đồ (Spearman, SSIM, trùng top-10% giữa `hm` và `hm_true`) và điểm số (hiệu HM đạo hàm đúng − HM) thay đổi bao nhiêu.
- **Mô tả thêm (không chấm).** Hàng dẫn đầu mỗi thước đo trong cả Bảng 1 mở rộng; hàng Gauge-X tốt nhất so với hàng HM tốt nhất và với hàng LayerCAM tốt nhất. Các hàng "tốt nhất" được chọn theo chính trung bình trên tập test, nên đây chỉ là mô tả.

**Attr. Loc. không xác định.** Quantus 0.6 trả NaN cho Attribution Localisation khi phần attribution trong mask lớn hơn tổng attribution, tức tỉ lệ $> 1$. Điều này chỉ xảy ra với bản đồ giữ dấu (hàng 1, 2, 4) khi phần ngoài mask có tổng âm; đó là quy ước của thước đo, không phải lỗi tính toán. Như cách HM lấy trung bình, trung bình của ô này lấy trên các ảnh xác định, và số ảnh không xác định của từng phương pháp được ghi ở §8. Hiệu ghép cặp dùng các ảnh xác định ở cả hai bên. Mọi ô NaN khác đều bị coi là lỗi: §2 và §7 kiểm lại từng ô NaN trên chính bản đồ.

**Hai kiểm cầu nối với nb01** bảo đảm phép ghép cặp có nghĩa:
- §5: chấm lại bản đồ LayerCAM (14) mà nb01 lưu bằng đường chấm của nb03, rồi so với điểm của nb01;
- §6–§7, trên **từng ảnh**: bản đồ `hm` của engine phải trùng bản đồ nb01 đã lưu cho cả 6 hàng Split, và ảnh đầu vào của nb01 phải trùng ảnh qua phép biến đổi của HM.

## 4. Engine và các phép kiểm

Engine nằm ở `scripts/gaugexai.py`. Năm phép kiểm dưới đây chạy trên ảnh test đầu tiên, đúng tensor đầu vào mà nb01 đã dùng; mọi phép phải đạt (cell có `assert`).

| | Phép kiểm | Cách làm | Ngưỡng |
| --- | --- | --- | --- |
| (i) | Mệnh đề 1–2, và đổi chỗ = Mệnh đề 3 | so với B của `dcpair.py` (nb02, đã khớp code HM tới $10^{-15}$), float64, CPU, maxpool lồi: σ và $a$ ở mọi chunk; bốn độ nhạy ở mọi chunk, cả code chính thức lẫn đạo hàm đúng | $10^{-12}$ |
| (ii) | bản đồ họ `hm` = bản đồ code HM | gọi đúng các hàm nb01 dùng (`_initialize_hybrid_module`, `_compute_explanation_with_module`) cho cả 6 hàng; float64 trên CPU và float32 trên thiết bị của kernel | $10^{-5}$ (float64) / $10^{-4}$ (float32) |
| (iii) | Mệnh đề 4 và bộ giải | $\Gamma_l(0) = 0$ đúng tuyệt đối trên lưới; tỉ số tại $t^\star$ bằng 1 | 0; $10^{-3}$ |
| (iv) | G0 = bản đồ cổ điển | so với Grad×Act và Saliency tính bằng autograd trên mô hình torchvision, độc lập với engine | $10^{-5}$ |
| (v) | float32 đủ | mọi bản đồ của mọi họ: float32 so với float64. Bản đồ kiểu gradient: sai lệch tương đối, in kèm sai số float32 của chính gradient cổ điển để đối chiếu. SplitLRP: tỉ lệ pixel lệch quá 1% max\|bản đồ\|, và 1 − Spearman | $10^{-3}$; $10^{-3}$ và $10^{-4}$ |

Ghi chú:
- Với (i), chunk `x` được bỏ qua vì hai bên ghi khác quy ước: `dcpair` ghi đầu ra $(x/2, -x/2)$ của chunk `x`, còn ta ghi cặp $(x, 0)$ mà conv đầu tiên đọc.
- Với (ii), code HM ép bản đồ về float32 ở bước cuối, nên sai lệch nhỏ nhất có thể ở mức khoảng $10^{-7}$.
- Với (v), SplitLRP được đo bằng tỉ lệ pixel thay vì chuẩn L2, vì quy tắc maxpool của LRP trong code HM định tuyến relevance về argmax của từng luồng. Ở một cửa sổ mà hai giá trị lớn nhất chênh nhau ít hơn sai số làm tròn của float32, ô thắng có thể khác nhau giữa float32 và float64. Khi đó relevance của cửa sổ đi sang pixel khác, nên bản đồ lệch ở vài pixel còn phần còn lại trùng nhau. Đây là tính gián đoạn của quy tắc LRP ở các điểm gần hoà, không phải float32 thiếu chính xác. Các bản đồ kiểu gradient chỉ gồm phép tính liên tục, nên vẫn dùng chuẩn L2.
- Phần float64 chạy trên CPU và mất vài phút. Đây là thời gian kiểm trước khi tốn giờ GPU ở §6.

In [ ]:
# (i) against method B of dcpair (nb02), float64, CPU, convex max-pool
X0, _, TARGET0, _ = load_test(0)
VGG64 = torchvision.models.vgg16(weights="IMAGENET1K_V1").eval().double()
NET64 = gx.SplitNet(VGG64)
DNET = dc.SeqNet.from_vgg(VGG64)
assert DNET.names == NET64.names
t0 = time.time()
B = dc.seq_method_B(DNET, X0, None, gx.THETA, gx.BIG_THETA, gx.ALPHA_HM, keep_pairs=True,
                    keep_sens=tuple(DNET.names), measure=False)
B_true = DNET.backward_B(B.cache, gx.ALPHA_HM, keep=tuple(DNET.names), conv_swap=False)
c64 = NET64.original(X0)
assert c64.target == B.cache.target
sig64, _ = NET64.sigma(c64, "hm", "convex")
fwd = max(max(gx.rel_diff(sig64[k + 1], B.pairs[n][0] + B.pairs[n][1]), gx.rel_diff(c64.a[k + 1], B.pairs[n][0] - B.pairs[n][1]))
          for k, n in enumerate(NET64.names) if n != "x")
bwd = {}
for swap, ref in ((True, B.sens), (False, B_true.kept)):
    g = NET64.gauge(c64, [gx.t_of_alpha(gx.ALPHA_HM)], NET64.names, "convex", swap=swap)
    worst = 0.0
    for k, n in enumerate(NET64.names):
        G, S, Dl = g[n]
        Dl = 0 if Dl is None else Dl
        D = c64.D[k]
        ours = (0.25 * (D + G + S + Dl), 0.25 * (-D - G + S + Dl), 0.25 * (-D + G + S - Dl), 0.25 * (D - G + S - Dl))
        worst = max(worst, max(gx.rel_diff(u, v) for u, v in zip(ours, ref[n])))
    bwd[swap] = worst
CHECKS = [dict(check="(i) lượt đi: σ = p + m và a = p − m ở mọi chunk, so với B của dcpair", value=fwd, tol=1e-12),
          dict(check="(i) lượt về, code chính thức (đổi Γ ↔ Δ sau mỗi conv): (pp, np, pn, nn) ở mọi chunk",
               value=bwd[True], tol=1e-12),
          dict(check="(i) lượt về, đạo hàm đúng: (pp, np, pn, nn) ở mọi chunk", value=bwd[False], tol=1e-12)]
del B, B_true, DNET
gc.collect()
print(f"(i) xong sau {time.time() - t0:.0f} s: lượt đi {fwd:.1e}, lượt về {bwd[True]:.1e} (chính thức), "
      f"{bwd[False]:.1e} (đạo hàm đúng)")

In [ ]:
# (ii) family 'hm' against the maps of the official HM code, float64 (CPU) and float32 (the kernel's device)
HM_CONFIGS = []
for f, rs in ROWS.groupby("hm_config", sort=False):
    cfg = json.loads((NB01_RUN / "configs" / "methods" / "fp32" / f).read_text())   # nb01's snapshot
    HM_CONFIGS.append((f, cfg, dict(zip(rs["id"], rs["hm_formula"]))))
FID = []
for dtype, model, device, net in (("float64", VGG64, torch.device("cpu"), NET64), ("float32", MODEL, DEVICE, NET)):
    ref, t0 = {}, time.time()
    for f, cfg, names in HM_CONFIGS:
        with hm_quiet():
            module, hcfg = _initialize_hybrid_module(copy.deepcopy(cfg), model, device, dtype, LOG)
            res = _compute_explanation_with_module(copy.deepcopy(cfg), module, X0.clone().to(device), TARGET0, hcfg,
                                                   model, LOG)
        for rid, fname in names.items():
            ref[rid] = np.asarray(res[fname], dtype=np.float32)
        del module
    hm_s, t0 = time.time() - t0, time.time()
    ex = gx.explain(net, X0, ROW_DICTS, families=("hm",), target=TARGET0)
    eng_s = time.time() - t0
    for r in ROWS.itertuples():
        FID.append(dict(row=r.id, kind=r.kind, layer=r.layer, pool=r.pool, hm_post=r.hm_post, dtype=dtype,
                        rel_diff=gx.rel_diff(gx.map224(ex.maps[(r.id, "hm")], r.hm_post), ref[r.id]),
                        tol=1e-5 if dtype == "float64" else 1e-4))
    print(f"(ii) {dtype}: code HM {hm_s:.0f} s, engine {eng_s:.1f} s cho cả 6 hàng")
FID = pd.DataFrame(FID)
FID["passed"] = FID["rel_diff"] <= FID["tol"]
FID.to_csv(DATA_DIR / "hm_fidelity.csv", index=False)
for dtype, g in FID.groupby("dtype", sort=False):
    CHECKS.append(dict(check=f"(ii) bản đồ họ hm = bản đồ code HM ({dtype}, 6 hàng)", value=g["rel_diff"].max(),
                       tol=g["tol"].iloc[0]))

In [ ]:
# (iii) Proposition 4 and the solver; (iv) G0 = classical maps; (v) float32 against float64
ALL_FAMILIES = ("hm", "hm_true", "gauge", "g0")
EX32 = gx.explain(NET, X0, ROW_DICTS, families=ALL_FAMILIES, target=TARGET0, keep=True)
gamma0 = max(float(EX32.gauge[p]["true"][l][0][0].abs().max()) for p in EX32.gauge for l in EX32.gauge[p]["true"])
ratio_dev = max(abs(d["ratio_star"] - 1.0) for d in EX32.diag if d["balanced"])
CHECKS += [dict(check="(iii) Γ_l(0) = 0 ở mọi tầng và mọi kiểu maxpool (hàng t = 0 của lưới)", value=gamma0, tol=0.0),
           dict(check="(iii) tỉ số ‖Γ_l(t*)‖ / ‖D_l‖ = 1 sau bộ giải (mọi tầng đạt cân bằng)", value=ratio_dev, tol=1e-3)]

# classical maps by autograd on a copy of the torchvision model, independent of the engine
clf = copy.deepcopy(MODEL)
for m in clf.modules():
    if isinstance(m, torch.nn.ReLU):
        m.inplace = False
acts = {}
def keep_input(name):
    def hook(mod, inp):
        inp[0].retain_grad()
        acts[name] = inp[0]
    return hook
grad_rows = ROWS[ROWS["kind"] != "lrp"]
hooks = [clf.features[int(l.split("_")[1])].register_forward_pre_hook(keep_input(l)) for l in grad_rows["layer"].unique()]
out = clf(X0.to(DEVICE))
out[0, EX32.target].backward()
for h in hooks:
    h.remove()
worst_g0 = 0.0
for r in grad_rows.itertuples():
    a, Dg = acts[r.layer].detach(), acts[r.layer].grad
    classical = {"cam_g": 0.5 * (Dg * a).sum(1), "grad": 0.5 * Dg.mean(1), "cam_pp": 0.125 * (Dg * a).sum(1)}[r.kind]
    worst_g0 = max(worst_g0, gx.rel_diff(EX32.maps[(r.id, "g0")], classical[0]))
CHECKS.append(dict(check="(iv) G0 = Grad×Act / Saliency tính bằng autograd trên mô hình torchvision", value=worst_g0, tol=1e-5))
del clf, acts, out

EX64 = gx.explain(NET64, X0, ROW_DICTS, families=ALL_FAMILIES, target=EX32.target)
kind_of, prec = dict(zip(ROWS["id"], ROWS["kind"])), []
for (rid, fam) in EX32.maps:
    m32, m64 = gx.map224(EX32.maps[(rid, fam)], "none"), gx.map224(EX64.maps[(rid, fam)], "none")
    prec.append(dict(row=rid, family=fam, kind=kind_of[rid], rel_diff=gx.rel_diff(m32, m64), spearman=gx.spearman(m32, m64),
                     off_pixels=float((np.abs(m32 - m64) > 0.01 * np.abs(m64).max()).mean())))
PREC = pd.DataFrame(prec)
PREC = PREC.merge(PREC[PREC["family"] == "g0"][["row", "rel_diff"]].rename(columns={"rel_diff": "classical_rel_diff"}),
                  on="row", how="left")
PREC.to_csv(DATA_DIR / "float_precision.csv", index=False)
grad_p, lrp_p = PREC[PREC["kind"] != "lrp"], PREC[PREC["kind"] == "lrp"]
CHECKS += [dict(check="(v) float32 so với float64, bản đồ kiểu gradient (cam_g, grad, cam_pp): sai lệch tương đối",
                value=grad_p["rel_diff"].max(), tol=1e-3),
           dict(check="(v) float32 so với float64, SplitLRP: tỉ lệ pixel lệch quá 1% max|bản đồ|",
                value=lrp_p["off_pixels"].max(), tol=1e-3),
           dict(check="(v) float32 so với float64, SplitLRP: 1 − Spearman", value=1.0 - lrp_p["spearman"].min(), tol=1e-4)]

CHK = pd.DataFrame(CHECKS)
CHK["passed"] = CHK["value"] <= CHK["tol"]
CHK.to_csv(DATA_DIR / "checks.csv", index=False)
del EX64, NET64, VGG64, EX32
gc.collect()
assert CHK["passed"].all(), CHK[~CHK["passed"]]
print(f"§4: {int(CHK['passed'].sum())}/{len(CHK)} phép kiểm đạt")

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
chk = pd.read_csv(DATA_DIR / "checks.csv")
display(rep.show_csv(RUN_DIR, "checks.csv", LABELS).style.format({LABELS["columns"]["value"]: "{:.2e}",
                                                                 LABELS["columns"]["tol"]: "{:.0e}"}))
display(rep.show_csv(RUN_DIR, "hm_fidelity.csv", LABELS).style.format({LABELS["columns"]["rel_diff"]: "{:.2e}",
                                                                      LABELS["columns"]["tol"]: "{:.0e}"}))
prec = rep.show_csv(RUN_DIR, "float_precision.csv", LABELS)
display(prec.style.format({LABELS["columns"]["rel_diff"]: "{:.1e}", "classical_rel_diff": "{:.1e}",
                           LABELS["columns"]["off_pixels"]: "{:.1e}", LABELS["columns"]["spearman"]: "{:.6f}"}))
display(Markdown(f"**Kết luận: {'mọi phép kiểm đạt' if chk['passed'].all() else 'CÓ PHÉP KIỂM KHÔNG ĐẠT'}** "
                 f"({int(chk['passed'].sum())}/{len(chk)})."))

## 5. Kiểm đường chấm điểm với nb01

Điểm của nb03 được tính bằng đúng `QuantusEvaluator` của HM, mỗi lần gọi một thước đo, như `BatchedQuantusEvaluator` mà nb01 dùng; tham số thước đo là bản chụp config của chính lần chạy nb01. Ảnh đầu vào, mask và lớp đích là các file nb01 đã lưu.

Phép kiểm: chấm lại bản đồ LayerCAM (14) mà nb01 đã lưu, trên `N_SCORE_CHECK` ảnh test đầu, bằng đường chấm của nb03, rồi so 4 thước đo tất định với điểm của nb01. Ngưỡng là sai lệch tương đối `SCORE_TOL`. Max Sens là thước đo ngẫu nhiên nên không có trong phép kiểm này. Nếu phép kiểm không đạt, notebook dừng trước khi tốn giờ GPU ở §6.

Trên CPU (chạy thử), Quantus chạy từng bước một, nên phép kiểm mất vài phút cho một ảnh.

In [ ]:
def score_det(image, maps, target, mask, metric_files):
    """The four deterministic metrics of `maps` ({name: 224 x 224 map}) by HM's QuantusEvaluator, as nb01."""
    res = {n: {} for n in maps}
    for f in metric_files:
        mc = json.loads(Path(f).read_text())
        if mc["metric_class"] == "max_sensitivity":
            continue
        with hm_quiet():
            ev = QuantusEvaluator(model=MODEL, device=str(DEVICE), metric_configs=[mc], softmax=False)
            out = ev.evaluate_single_image(image=image, explanations=maps, target_class=target, segmentation_mask=mask,
                                           explain_func=None, logger=LOG)
        for n, v in out.items():
            res[n].update(v)
    return res

SC, t0 = [], time.time()
nb01_vals = NB01_LONG[NB01_LONG["row"] == 7].set_index(["image_name", "metric"])["value"]
for k in range(N_SCORE_CHECK):
    x, mask, target, _ = load_test(k)
    lc = np.load(IMAGES[k]["nb01_layercam14"])["explanation"]
    ours = score_det(x.to(DEVICE), {"layercam14": lc}, target, mask, NB01_METRIC_FILES)["layercam14"]
    for key in ("selectivity", "attribution_localisation", "pointing_game", "pf_auc5", "pf_auc20"):
        theirs = float(nb01_vals.loc[(IMAGES[k]["image_name"], key)])
        mine = float(ours[rep.RESULT_COLUMNS[key]])
        SC.append(dict(image=k, metric=key, ours=mine, nb01=theirs,
                       rel_diff=abs(mine - theirs) / max(abs(theirs), 1e-12), tol=SCORE_TOL))
SC = pd.DataFrame(SC)
SC["passed"] = SC["rel_diff"] <= SC["tol"]
SC.to_csv(DATA_DIR / "scoring_check.csv", index=False)
assert SC["passed"].all(), SC[~SC["passed"]]
print(f"§5: {N_SCORE_CHECK} ảnh × 5 ô khớp nb01 (sai lệch tương đối lớn nhất {SC['rel_diff'].max():.1e}) "
      f"sau {time.time() - t0:.0f} s")

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
sc = pd.read_csv(DATA_DIR / "scoring_check.csv")
display(rep.show_csv(RUN_DIR, "scoring_check.csv", LABELS).style.format({LABELS["columns"]["rel_diff"]: "{:.1e}"}))
display(Markdown(f"**Đường chấm điểm: {'KHỚP nb01' if sc['passed'].all() else 'LỆCH nb01'}** "
                 f"({int(sc['passed'].sum())}/{len(sc)} ô trong ngưỡng)."))

## 6. Chạy song song: bản đồ, cầu nối, 6 thước đo

Cell đầu ghi `run_config.json` và định nghĩa `run_workers`. Hàm này mở `N_WORKERS` tiến trình `scripts/nb03_worker.py`, `WORKERS_PER_GPU` tiến trình trên mỗi GPU trong `GPUS`. Mỗi tiến trình chỉ thấy GPU của mình và dùng `CPU_THREADS` luồng CPU.

**Chia việc động.** Worker rảnh nhận ảnh $k$ kế tiếp chưa có kết quả, bằng cách tạo nguyên tử file `workers/claims/image_KKK`. GPU nhanh hơn tự làm nhiều ảnh hơn, và không có rào chắn giữa các pha: các worker ở các giai đoạn khác nhau, nên GPU luôn có việc.

**Mỗi ảnh $k$ đi trọn một mạch trong một worker:**
1. nạp tensor đầu vào, mask và lớp đích mà nb01 đã dùng;
2. bản đồ thô của 6 hàng cho ba họ `hm`, `hm_true`, `gauge`, kèm $\alpha^\star$ và đường tỉ số của quy tắc cân bằng;
3. kiểm cầu nối: bản đồ `hm` so với bản đồ nb01 đã lưu; so bản đồ giữa các họ (§11);
4. 4 thước đo tất định cho 11 biến thể, mỗi thước đo một lần gọi `QuantusEvaluator` cho mọi biến thể;
5. Max Sens cho 11 biến thể. Mọi biến thể bắt đầu từ cùng seed $42 \cdot 10^4 + k$, nên gặp cùng các ảnh nhiễu. Engine tính mỗi ảnh nhiễu **một lần** cho cả 6 hàng và cả hai họ, rồi dùng lại.

**Ghi kết quả.** File `data/scores/image_KKK.csv` được ghi sau cùng và đánh dấu ảnh đã xong. Chạy lại sẽ bỏ qua các ảnh đã xong, nên một lần chạy bị ngắt có thể tiếp tục. Nếu một ảnh lỗi, worker ghi lỗi vào `workers/errors/` rồi làm ảnh tiếp theo; cell dừng và in lỗi sau khi mọi worker xong. Thanh tiến trình đếm số ảnh đã xong.

In [ ]:
SNAP = RUN_DIR / "configs" / "metrics"
SNAP.mkdir(parents=True, exist_ok=True)
METRIC_FILES = []
for src in NB01_METRIC_FILES:
    mc = json.loads(src.read_text())
    if SMOKE and not GPUS:                        # smoke test on the CPU: coarser Pixel Flipping / Selectivity
        mc["params"].update(CPU_METRIC_PARAMS.get(mc["metric_class"], {}))
    (SNAP / src.name).write_text(json.dumps(mc, indent=2))
    METRIC_FILES.append(str(SNAP / src.name))
RUN_CONFIG = {
    "n_images": N_IMAGES, "images": IMAGES, "split_seed": SPLIT_SEED, "hm_repo": str(HM_REPO),
    "imagenet": str(IMAGENET), "imagenet_s": str(IMAGENET_S), "rows": ROW_DICTS, "variants": VARIANTS,
    "map_families": MAP_FAMILIES, "pairs": PAIRS, "metric_files": METRIC_FILES, "t_grid": T_GRID.tolist(),
    "quantus_chunk": QUANTUS_CHUNK, "tf32": TF32, "data_dir": str(DATA_DIR), "cpu_threads": CPU_THREADS,
}
(RUN_DIR / "run_config.json").write_text(json.dumps(RUN_CONFIG, indent=1, ensure_ascii=False), encoding="utf-8")
# a rerun resumes from the images already done: refuse it if those were computed with another configuration
import hashlib
FINGERPRINT = hashlib.sha1(json.dumps(dict(
    images=[(r["image_name"], r["target"]) for r in IMAGES], rows=ROW_DICTS, variants=VARIANTS, families=MAP_FAMILIES,
    t_grid=T_GRID.tolist(), chunk=QUANTUS_CHUNK, tf32=TF32, metrics=[json.loads(Path(f).read_text()) for f in METRIC_FILES]),
    sort_keys=True).encode()).hexdigest()[:12]
fp_file = DATA_DIR / "config_fingerprint.txt"
if fp_file.exists() and fp_file.read_text().strip() != FINGERPRINT and any(SCORE_DIR.glob("image_*.csv")):
    raise RuntimeError(f"{DATA_DIR} đã có kết quả tính với cấu hình khác ({fp_file.read_text().strip()} ≠ {FINGERPRINT}): "
                       f"chuyển thư mục đó đi rồi chạy lại")
fp_file.write_text(FINGERPRINT)
WORKER_DIR = RUN_DIR / "workers"
CLAIMS, ERRORS = WORKER_DIR / "claims", WORKER_DIR / "errors"
for d in (WORKER_DIR, CLAIMS, ERRORS):
    d.mkdir(parents=True, exist_ok=True)

def n_done():
    return sum((SCORE_DIR / f"image_{k:03d}.csv").exists() for k in range(N_IMAGES))

def run_workers(label):
    """Start the workers (WORKERS_PER_GPU on every GPU of GPUS, one on the CPU without GPU), draw one bar for all
    of them, wait; raise with the log tail if a worker or an image failed."""
    for f in [*CLAIMS.glob("image_*"), *ERRORS.glob("image_*")]:   # no worker runs now: old claims are stale
        f.unlink()
    slots = [g for _ in range(WORKERS_PER_GPU) for g in GPUS][:N_WORKERS] if GPUS else [None]
    threads = str(CPU_THREADS)
    procs, logs = [], []
    for w, g in enumerate(slots):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES="" if g is None else str(g), PYTHONIOENCODING="utf-8",
                   OMP_NUM_THREADS=threads, MKL_NUM_THREADS=threads, OPENBLAS_NUM_THREADS=threads)
        logs.append(open(WORKER_DIR / f"w{w}.log", "w", encoding="utf-8"))
        procs.append(subprocess.Popen([sys.executable, str(EXP_DIR / "scripts" / "nb03_worker.py"), str(RUN_DIR), str(w)],
                                      stdout=logs[-1], stderr=subprocess.STDOUT, env=env, cwd=EXP_DIR))
        time.sleep(1)                             # stagger the start-up (dataset index, model weights)
    start = n_done()
    with Progress(RUN_DIR, label, N_IMAGES, unit="ảnh") as bar:
        bar.update(start)
        while True:
            running = any(p.poll() is None for p in procs)
            bar.update(n_done() - bar.done)
            if not running:
                break
            time.sleep(3)
    for f in logs:
        f.close()
    bad = [w for w, p in enumerate(procs) if p.returncode != 0]
    failed = sorted(ERRORS.glob("image_*.txt"))
    if bad or failed:
        tail = (WORKER_DIR / f"w{bad[0]}.log").read_text(encoding="utf-8")[-3000:] if bad else \
            failed[0].read_text(encoding="utf-8")[-3000:]
        raise RuntimeError(f"worker lỗi: {bad}; ảnh lỗi: {[f.stem for f in failed]}; chạy lại cell này để tính tiếp "
                           f"các ảnh còn thiếu. Log: {WORKER_DIR}\n{tail}")
    return n_done() - start

if DEVICE.type == "cuda":                         # leave the GPU to the workers: from §7 on the kernel needs no model
    NET, MODEL = None, MODEL.cpu()
    gc.collect()
    torch.cuda.empty_cache()
t0 = time.time()
new = run_workers("§6 tính trên ảnh test")
wall = time.time() - t0
RUN_LOG = DATA_DIR / "run_log.csv"
log_rows = pd.read_csv(RUN_LOG).to_dict("records") if RUN_LOG.exists() else []
log_rows.append(dict(time=time.strftime("%Y-%m-%d %H:%M:%S"), images_new=new, wall_s=wall, n_workers=N_WORKERS,
                     gpus=",".join(map(str, GPUS)) or "cpu", workers_per_gpu=WORKERS_PER_GPU if GPUS else 1))
pd.DataFrame(log_rows).to_csv(RUN_LOG, index=False)
print(f"§6: {new} ảnh mới, {n_done()}/{N_IMAGES} ảnh xong, {wall / 60:.1f} phút với {N_WORKERS} worker")

## 7. Gom kết quả và kiểm đủ

Gom các file theo ảnh thành:
- `scores_long.csv`: điểm của 11 biến thể trên từng ảnh (dạng của HM);
- `images.csv`, `alpha_star.csv`, `bridge.csv`, `swap_effect.csv`, `timing.csv`: thông tin ảnh, $\alpha^\star$, kiểm cầu nối, độ tương đồng bản đồ, thời gian;
- `ratio_curves.npz`: đường tỉ số của mọi ảnh;
- `fig_images.npz`: ảnh và mask cho các hình minh hoạ.

Code HM ghi NaN khi một thước đo lỗi, nên cell dừng nếu có ô NaN, thay vì lặng lẽ lấy trung bình trên phần còn lại. Ngoại lệ duy nhất là Attr. Loc. không xác định theo định nghĩa của Quantus (§3.7): mỗi ô Attr. Loc. NaN được kiểm lại trên bản đồ đã lưu. Kiểm cầu nối thì không dừng notebook: kết quả được ghi vào `bridge.csv` và đưa vào kết luận ở §14.

In [ ]:
assert n_done() == N_IMAGES, f"mới có {n_done()}/{N_IMAGES} ảnh: chạy lại §6"
SCORES = pd.concat([pd.read_csv(SCORE_DIR / f"image_{k:03d}.csv") for k in range(N_IMAGES)], ignore_index=True)
cols = [rep.RESULT_COLUMNS[k] for k in rep.METRIC_KEYS]
missing_cols = [c for c in cols if c not in SCORES.columns]
assert not missing_cols, f"thước đo thiếu cột {missing_cols}: xem log trong {RUN_DIR / 'workers'}"
AL = rep.RESULT_COLUMNS["attribution_localisation"]
nan_other = {c: int(SCORES[c].isna().sum()) for c in cols if c != AL}
nan_al = SCORES[SCORES[AL].isna()]
masks, bad_al = {}, []
for r in nan_al.itertuples():                     # NaN by Quantus' definition (inside / total > 1), or a failure?
    masks.setdefault(r.image, load_test(r.image)[1])
    raw_map = torch.from_numpy(np.load(MAP_DIR / f"image_{r.image:03d}.npz")[f"{r.row}|{r.family}"])
    if not attrloc_undefined(gx.map224(raw_map, r.post), masks[r.image]):
        bad_al.append((r.image, r.row, r.family))
assert not any(nan_other.values()) and not bad_al, \
    f"ô NaN do lỗi: {nan_other}, Attr. Loc. {bad_al[:5]}: xem log trong {RUN_DIR / 'workers'}"
assert len(SCORES) == N_IMAGES * len(VARIANTS)
SCORES.to_csv(DATA_DIR / "scores_long.csv", index=False)

img_recs, alpha_recs, bridge_recs, sim_recs, tim_recs, curves, KEYS = [], [], [], [], [], [], None
for k in range(N_IMAGES):
    p = json.loads((PER_IMAGE_DIR / f"image_{k:03d}.json").read_text(encoding="utf-8"))
    img_recs.append(p["meta"])
    alpha_recs += [dict(image=k, **a) for a in p["alpha"]]
    bridge_recs += [dict(image=k, **b) for b in p["bridge"]]
    sim_recs += [dict(image=k, **s) for s in p["similarity"]]
    tim_recs.append(dict(image=k, worker=p["meta"]["worker"], gpu=p["meta"]["gpu"] or "cpu",
                         peak_gib=p["meta"]["peak_gib"], **p["timing"]))
    z = np.load(MAP_DIR / f"image_{k:03d}.npz")
    keys = sorted(f for f in z.files if f.startswith("ratio|"))
    KEYS = KEYS or keys
    assert keys == KEYS
    curves.append(np.stack([z[f] for f in keys]))
IMG = pd.DataFrame(img_recs)
IMG["target_matches_own_pred"] = IMG["target"] == IMG["own_pred"]
IMG.to_csv(DATA_DIR / "images.csv", index=False)
pd.DataFrame(alpha_recs).to_csv(DATA_DIR / "alpha_star.csv", index=False)
BR = pd.DataFrame(bridge_recs)
BR["passed"] = np.where(BR["kind"] == "lrp",
                        (BR["off_pixels"] <= BRIDGE_TOL["off_pixels"]) & (1 - BR["spearman"] <= BRIDGE_TOL["one_minus_spearman"]),
                        BR["rel_diff"] <= BRIDGE_TOL["rel_diff"])
BR.to_csv(DATA_DIR / "bridge.csv", index=False)
pd.DataFrame(sim_recs).to_csv(DATA_DIR / "swap_effect.csv", index=False)
pd.DataFrame(tim_recs).to_csv(DATA_DIR / "timing.csv", index=False)
np.savez_compressed(DATA_DIR / "ratio_curves.npz", ts=T_GRID, keys=np.array([k.split("|", 1)[1] for k in KEYS]),
                    curves=np.stack(curves))
fig_images = {}
for k in range(N_FIG):
    raw = load_test(k)[3]
    fig_images[f"image_{k}"] = np.asarray(T.CenterCrop(224)(T.Resize(256)(raw)))
    m = load_test(k)[1]
    fig_images[f"mask_{k}"] = m[..., 0] if m.ndim == 3 else m
np.savez_compressed(DATA_DIR / "fig_images.npz", **fig_images)
BRIDGE_OK = bool(BR["passed"].all() and len(BR) == N_IMAGES * len(ROWS) and (IMG["input_max_abs_diff"] < 1e-5).all())
print(f"§7: {len(SCORES)} dòng điểm, {N_IMAGES} ảnh ({len(nan_al)} ô Attr. Loc. không xác định theo Quantus); "
      f"cầu nối: {int(BR['passed'].sum())}/{len(BR)} (ảnh, hàng) đạt; "
      f"lớp nb01 = lớp mô hình dự đoán ở {int(IMG['target_matches_own_pred'].sum())}/{N_IMAGES} ảnh")

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
display(rep.bridge_table(RUN_DIR, LABELS).style.format({LABELS["columns"]["worst_rel_diff"]: "{:.1e}",
                                                       LABELS["columns"]["worst_off_pixels"]: "{:.1e}",
                                                       LABELS["columns"]["worst_spearman"]: "{:.6f}"}))
img = pd.read_csv(DATA_DIR / "images.csv")
display(Markdown(f"Ảnh đầu vào của nb01 so với ảnh qua phép biến đổi của HM: sai lệch tuyệt đối lớn nhất "
                 f"{img['input_max_abs_diff'].max():.1e}. Lớp được giải thích (của nb01) trùng lớp mô hình dự đoán "
                 f"lại ở {int(img['target_matches_own_pred'].sum())}/{len(img)} ảnh."))

## 8. Bảng 1 mở rộng

Mỗi hàng của bảng là một phương pháp:
- 9 hàng của nb01: 6 hàng Split của HM (code chính thức), LayerCAM tầng 14 và 2, Random;
- 5 hàng HM với đạo hàm đúng và 6 hàng Gauge-X của nb03.

Mỗi ô là trung bình ± SE theo ảnh, với $\mathrm{SE} = s/\sqrt n$. Các hàng Split được xếp theo từng hàng của Bảng 1: HM, HM đạo hàm đúng, Gauge-X. Mọi số liệu ở dạng dài nằm ở `all_long.csv` (ảnh × phương pháp × thước đo).

Ở Attr. Loc., trung bình lấy trên các ảnh xác định (§3.7). Bảng thứ hai ghi số ảnh không xác định của từng phương pháp, nếu có.

In [ ]:
names = [r["image_name"] for r in IMAGES]
new_long = []
for rid, fam, post in VARIANTS:
    g = SCORES[(SCORES["row"] == rid) & (SCORES["family"] == fam)]
    for key in rep.METRIC_KEYS:
        new_long.append(pd.DataFrame({"image_name": g["image_name"].values, "method": f"{fam}_{rid}", "metric": key,
                                      "value": g[rep.RESULT_COLUMNS[key]].astype(float).values}))
old = NB01_LONG[NB01_LONG["image_name"].isin(names)]
old = old.assign(method="nb01_r" + old["row"].astype(str))[["image_name", "method", "metric", "value"]]
ALL = pd.concat([old, *new_long], ignore_index=True)
ALL.to_csv(DATA_DIR / "all_long.csv", index=False)
SUMMARY = ALL.groupby(["method", "metric"])["value"].agg(mean="mean", sd="std", n="count").reset_index()
SUMMARY["se"] = SUMMARY["sd"] / np.sqrt(SUMMARY["n"])
SUMMARY.to_csv(DATA_DIR / "summary.csv", index=False)
full = (SUMMARY["n"] == N_IMAGES) | (SUMMARY["metric"] == "attribution_localisation") | \
       ((SUMMARY["method"] == "nb01_r9") & (SUMMARY["metric"] == "max_sensitivity"))
assert full.all(), SUMMARY[~full]
UND = ALL[ALL["metric"] == "attribution_localisation"].groupby("method")["value"].agg(
    n_undefined=lambda v: int(v.isna().sum()), n="size").reset_index()
UND[UND["n_undefined"] > 0].to_csv(DATA_DIR / "undefined_attrloc.csv", index=False)
print(f"§8: {SUMMARY['method'].nunique()} phương pháp × {SUMMARY['metric'].nunique()} thước đo, {N_IMAGES} ảnh")

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
display(Markdown("**Bảng 1 mở rộng: trung bình ± SE theo ảnh.**"
                 + (" *Chạy thử: số liệu chưa có giá trị kết luận.*" if SMOKE else "")))
display(rep.table1(RUN_DIR, LABELS))
und = pd.read_csv(DATA_DIR / "undefined_attrloc.csv")
display(Markdown("**Attr. Loc. không xác định (Quantus trả NaN khi tỉ lệ trong mask > 1):** "
                 + ("không có." if und.empty else "")))
if not und.empty:
    display(rep.show_csv(RUN_DIR, "undefined_attrloc.csv", LABELS))
rep.plot_scores(RUN_DIR, LABELS); plt.show()

## 9. So sánh ghép cặp theo ảnh (H2)

Ba phép so sánh, ở từng hàng Split và từng thước đo:
- **Gauge-X − HM (chính thức)**: phép so sánh chính của H2;
- **Gauge-X − HM (đạo hàm đúng)**: tách tác dụng của quy tắc cân bằng ($\alpha^\star_l$ và σ cân bằng) khỏi tác dụng của việc sửa lỗi đổi chỗ;
- **HM (đạo hàm đúng) − HM (chính thức)**: tác dụng của riêng việc sửa lỗi đổi chỗ lên điểm số (H4).

Mỗi ô là $\bar d \pm \mathrm{SE}$ và dấu `+`, `=`, `−` theo hướng tốt của thước đo (§3.7). Ở thước đo có mũi tên ↓, hiệu âm là tốt hơn. Hình nhiệt tô màu theo $z = \bar d/\mathrm{SE}$ đã đổi về hướng tốt: xanh là tốt hơn, đỏ là kém hơn.

Bảng cuối là phần mô tả: hàng dẫn đầu ở từng thước đo, và hàng Gauge-X tốt nhất so với hàng HM tốt nhất và hàng LayerCAM tốt nhất.

In [ ]:
from scipy.stats import norm
WIDE = ALL.pivot_table(index=["metric", "image_name"], columns="method", values="value")

def paired(metric, a, b):
    d = (WIDE.loc[metric][a] - WIDE.loc[metric][b]).dropna()
    se = d.std(ddof=1) / np.sqrt(len(d)) if len(d) > 1 else np.nan
    return float(d.mean()), float(se), len(d)

COMPS = {"gauge-hm": ("gauge", "nb01"), "gauge-hm_true": ("gauge", "hm_true"), "hm_true-hm": ("hm_true", "nb01")}
P = []
for comp, (a, b) in COMPS.items():
    for r in ROWS.itertuples():
        if f"{a}_{r.id}" not in WIDE.columns or f"{b}_{r.id}" not in WIDE.columns:
            continue
        for key in rep.METRIC_KEYS:
            m, se, n = paired(key, f"{a}_{r.id}", f"{b}_{r.id}")
            z = rep.DIRECTION[key] * m / se if np.isfinite(se) and se > 0 else np.nan
            P.append(dict(comparison=comp, row=r.id, metric=key, mean=m, se=se, n=n, z=z,
                          mark=rep.mark(m, se, rep.DIRECTION[key])))
P = pd.DataFrame(P)
P.to_csv(DATA_DIR / "paired.csv", index=False)

# descriptive: leader of every metric, best Gauge-X row against the best HM row and the best LayerCAM row
means = SUMMARY.pivot(index="method", columns="metric", values="mean")
best = []
for key in rep.METRIC_KEYS:
    s = rep.DIRECTION[key] * means[key].dropna()           # larger = better
    bg = s[[m for m in s.index if m.startswith("gauge_")]].idxmax()
    bh = s[[f"nb01_r{i}" for i in range(1, 7)]].idxmax()
    bl = s[["nb01_r7", "nb01_r8"]].idxmax()
    m1, e1, _ = paired(key, bg, bh)
    m2, e2, _ = paired(key, bg, bl)
    best.append(dict(metric=key, leader=s.idxmax(), best_gauge=bg, best_hm=bh, vs_hm_mean=m1, vs_hm_se=e1,
                     vs_hm_mark=rep.mark(m1, e1, rep.DIRECTION[key]), best_layercam=bl, vs_layercam_mean=m2,
                     vs_layercam_se=e2, vs_layercam_mark=rep.mark(m2, e2, rep.DIRECTION[key])))
pd.DataFrame(best).to_csv(DATA_DIR / "best_rows.csv", index=False)

H2 = P[P["comparison"] == "gauge-hm"]
Z_BONF = float(norm.ppf(1 - 0.025 / H2_CELLS))
H2_COUNTS = {k: int((H2["mark"] == k).sum()) for k in ("+", "=", "−", "?")}
H2_BONF_WORSE = int((H2["z"] < -Z_BONF).sum())
print(f"§9: Gauge-X − HM: {H2_COUNTS['+']} ô +, {H2_COUNTS['=']} ô =, {H2_COUNTS['−']} ô −"
      f"{', ' + str(H2_COUNTS['?']) + ' ô ? (không có SE)' if H2_COUNTS['?'] else ''}; "
      f"kém vượt ngưỡng Bonferroni (z < −{Z_BONF:.2f}): {H2_BONF_WORSE} ô")

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
for comp in rep.COMPARISONS:
    display(Markdown(f"**{LABELS['comparisons'][comp]}**: hiệu ghép cặp theo ảnh ± SE; `+` tốt hơn, `−` kém hơn quá 2·SE."))
    display(rep.paired_table(RUN_DIR, LABELS, comp))
rep.plot_paired(RUN_DIR, LABELS); plt.show()
display(Markdown("**Mô tả: hàng dẫn đầu và các hàng tốt nhất** (chọn theo chính trung bình trên tập test)."))
display(rep.best_table(RUN_DIR, LABELS))

## 10. α* của quy tắc cân bằng (H1)

$\alpha^\star_l$ được giải trên từng ảnh test (ảnh gốc, chưa thêm nhiễu), ở mọi tổ hợp (maxpool, tầng) mà các hàng dùng. Hình thứ hai là đường tỉ số $\lVert\Gamma_l(t)\rVert / \lVert D_l\rVert$ trên lưới, trung vị và khoảng tứ phân vị qua các ảnh. Vạch dọc đứt là $t = 0.2$, tức α = 0.4 của HM.

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
display(rep.alpha_table(RUN_DIR, LABELS, band=H1_BAND))
rep.plot_alpha(RUN_DIR, LABELS, band=H1_BAND); plt.show()
rep.plot_ratio_curves(RUN_DIR, LABELS); plt.show()
a = rep.read_alpha(RUN_DIR)
h1 = a[a["derivative"] == "true"].groupby(["pool", "layer"])["alpha_star"].mean()
h1 = h1[[c in rep.HM_COMBOS for c in h1.index]]
ok = len(h1) == len(rep.HM_COMBOS) and bool(((h1 >= H1_BAND[0]) & (h1 <= H1_BAND[1])).all())
display(Markdown(f"**H1: {'ĐẠT' if ok else 'KHÔNG ĐẠT'}.** α* trung bình (đạo hàm đúng) ở 5 tổ hợp HM dùng: "
                 + ", ".join(f"{LABELS['pools'][p]} {l.replace('features_', 'tầng ')}: {v:.3f}" for (p, l), v in h1.items())
                 + f"; dải H1 = [{H1_BAND[0]}, {H1_BAND[1]}]."))

## 11. Lỗi đổi chỗ ở conv: bản đồ (H4)

Cùng α = 0.4 và θ, Θ, hai họ `hm` (code chính thức) và `hm_true` (đạo hàm đúng) chỉ khác nhau ở lượt về Conv2d. Ta so bản đồ của chúng trên từng ảnh, mỗi hàng với hậu xử lý của HM, bằng ba độ đo:
- **Spearman**: tương quan hạng của các pixel;
- **SSIM**: sau khi đưa mỗi bản đồ về $[0, 1]$, như các phép kiểm ngẫu nhiên hoá;
- **trùng top-10%**: tỉ lệ pixel thuộc 10% lớn nhất của cả hai bản đồ.

Các độ đo được tính ở độ phân giải gốc của tầng. Bảng có thêm Gauge-X so với từng họ HM. Phần điểm số của H4 là bảng HM (đạo hàm đúng) − HM ở §9. SplitLRP không bị ảnh hưởng bởi lỗi này, nên chỉ có cặp Gauge-X ~ HM.

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
display(rep.swap_table(RUN_DIR, LABELS))
rep.plot_swap(RUN_DIR, LABELS); plt.show()
s = pd.read_csv(DATA_DIR / "swap_effect.csv")
m = s[s["pair"] == "hm~hm_true"].groupby("row")["spearman"].mean()
display(Markdown("**H4 (đo):** Spearman trung bình giữa code chính thức và đạo hàm đúng: "
                 + ", ".join(f"{LABELS['rows'][r]}: {v:.2f}" for r, v in m.items()) + "."))

## 12. Bản đồ minh hoạ

Mỗi hình là một ảnh test. Mỗi dòng là một hàng, mỗi cột là một họ, và mỗi bản đồ dùng đúng hậu xử lý của hàng HM tương ứng. Cột HM (code chính thức) là bản đồ của engine, trùng bản đồ nb01 (kiểm cầu nối ở §7). Bản đồ giữ dấu dùng thang màu phân kỳ, chuẩn hoá theo $\max\lvert\cdot\rvert$ (đỏ dương, xanh âm, trắng là 0). Bản đồ $\lvert\cdot\rvert$ dùng thang một màu. Đường trắng trên ảnh gốc là mask của ImageNet-S. Ô "= HM" ở SplitLRP: họ `hm_true` trùng `hm`.

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
for k in range(N_FIG):
    rep.plot_maps(RUN_DIR, LABELS, image=k); plt.show()

## 13. Thời gian, bộ nhớ, thông lượng

- Bảng đầu: thời gian mỗi ảnh của từng giai đoạn trong worker (trung vị, trung bình, lớn nhất). Các worker trên cùng một GPU chạy đồng thời, nên đây là thời gian đồng hồ của một ảnh khi GPU được chia sẻ, không phải thời gian khi chạy một mình.
- Bảng thứ hai: số ảnh, thời gian trung bình mỗi ảnh và đỉnh bộ nhớ GPU của từng worker. Đỉnh bộ nhớ dùng để chỉnh `WORKERS_PER_GPU` cho vừa bộ nhớ GPU.
- Bảng cuối: thời gian đồng hồ của §6 và thông lượng (ảnh/giờ). Lần chạy thử thì ngoại suy lên 566 ảnh với cùng số worker; ngoại suy này bi quan vì chi phí khởi động chiếm phần lớn khi chỉ có ít ảnh.

In [ ]:
runs = pd.read_csv(DATA_DIR / "run_log.csv")
fresh = runs[runs["images_new"] > 0]
last = fresh.iloc[-1] if len(fresh) else runs.iloc[-1]
per_image_wall = last["wall_s"] / max(int(last["images_new"]), 1)
tim = pd.read_csv(DATA_DIR / "timing.csv")
EST = pd.DataFrame([
    dict(quantity="ảnh mới ở lần chạy §6 gần nhất", value=int(last["images_new"])),
    dict(quantity="thời gian đồng hồ của lần đó (phút)", value=last["wall_s"] / 60),
    dict(quantity="số worker / GPU", value=f"{int(last['n_workers'])} / {last['gpus']}"),
    dict(quantity="thông lượng (ảnh/giờ)", value=3600 / per_image_wall),
    dict(quantity="tổng thời gian tính của mọi worker (giờ)", value=tim["total"].sum() / 3600),
    dict(quantity="ước lượng đồng hồ cho 566 ảnh, cùng số worker (giờ)", value=per_image_wall * N_TEST / 3600),
])
EST.to_csv(DATA_DIR / "runtime_estimate.csv", index=False)
print(f"§13: {3600 / per_image_wall:.0f} ảnh/giờ với {int(last['n_workers'])} worker")

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
display(rep.timing_table(RUN_DIR, LABELS))
display(rep.gpu_table(RUN_DIR, LABELS))
display(rep.show_csv(RUN_DIR, "runtime_estimate.csv", LABELS))

## 14. Kết luận

Cell tính toán gom các kết luận vào `nb03_verdict.json` và `nb03_verdict.md`:
- các phép kiểm engine (§4) và đường chấm điểm (§5): phải đạt hết, nếu không notebook đã dừng;
- kiểm cầu nối trên từng ảnh (§7);
- H2 theo quy tắc ở §3.7, kèm số ô kém vượt ngưỡng Bonferroni;
- H1;
- H4: chỉ đo, gồm Spearman bản đồ và số ô điểm số thay đổi quá 2·SE.

Lần chạy thử vẫn ghi đủ các file, kèm dòng cảnh báo rằng số liệu chưa có giá trị kết luận. Với 1 ảnh không tính được SE, nên H2 được ghi là "không chấm được".

In [ ]:
chk = pd.read_csv(DATA_DIR / "checks.csv")
sc = pd.read_csv(DATA_DIR / "scoring_check.csv")
br = pd.read_csv(DATA_DIR / "bridge.csv")
a = rep.read_alpha(RUN_DIR)
h1 = a[a["derivative"] == "true"].groupby(["pool", "layer"])["alpha_star"].mean()
h1 = h1[[c in rep.HM_COMBOS for c in h1.index]]
assert len(h1) == len(rep.HM_COMBOS), f"thiếu α* ở {set(rep.HM_COMBOS) - set(h1.index)}"
h1_ok = bool(((h1 >= H1_BAND[0]) & (h1 <= H1_BAND[1])).all())
p = pd.read_csv(DATA_DIR / "paired.csv")
h2 = p[p["comparison"] == "gauge-hm"]
h4 = p[p["comparison"] == "hm_true-hm"]
sw = pd.read_csv(DATA_DIR / "swap_effect.csv")
sw = sw[sw["pair"] == "hm~hm_true"].groupby("row")["spearman"].mean()
counts = {k: int((h2["mark"] == k).sum()) for k in ("+", "=", "−", "?")}
h2_verdict = ("KHÔNG CHẤM ĐƯỢC (1 ảnh: không tính được SE)" if counts["?"] == len(h2)
              else "ĐẠT" if counts["−"] == 0 and counts["?"] == 0 else "KHÔNG ĐẠT")
worse = h2[h2["mark"] == "−"]
V = [
    dict(hypothesis="§4 phép kiểm engine", verdict="ĐẠT" if chk["passed"].all() else "KHÔNG ĐẠT",
         detail=f"{int(chk['passed'].sum())}/{len(chk)} phép kiểm"),
    dict(hypothesis="§5 đường chấm điểm so với nb01", verdict="KHỚP" if sc["passed"].all() else "LỆCH",
         detail=f"{int(sc['passed'].sum())}/{len(sc)} ô, sai lệch tương đối lớn nhất {sc['rel_diff'].max():.1e}"),
    dict(hypothesis="§7 cầu nối: bản đồ hm của engine = bản đồ nb01, từng ảnh", verdict="ĐẠT" if BRIDGE_OK else "KHÔNG ĐẠT",
         detail=f"{int(br['passed'].sum())}/{len(br)} (ảnh, hàng); sai lệch L2 lớn nhất (hàng gradient) "
                f"{br.loc[br['kind'] != 'lrp', 'rel_diff'].max():.1e}"),
    dict(hypothesis="H2: Gauge-X không kém HM (chính thức) ở 36 ô (hàng × thước đo)", verdict=h2_verdict,
         detail=f"{counts['+']} ô +, {counts['=']} ô =, {counts['−']} ô −; kém vượt ngưỡng Bonferroni "
                f"(z < −{Z_BONF:.2f}): {H2_BONF_WORSE} ô"
                + ("; ô kém: " + ", ".join(f"{LABELS['rows'][r.row]} / {LABELS['metrics'][r.metric]} (z = {r.z:.1f})"
                                          for r in worse.itertuples()) if len(worse) else "")),
    dict(hypothesis="H1: α* trung bình ∈ [0.33, 0.42] ở 5 tổ hợp HM dùng", verdict="ĐẠT" if h1_ok else "KHÔNG ĐẠT",
         detail=", ".join(f"{p_} {l.split('_')[1]}: {v:.3f}" for (p_, l), v in h1.items())),
    dict(hypothesis="H4: code chính thức so với đạo hàm đúng (đo)", verdict="đo",
         detail="Spearman bản đồ: " + ", ".join(f"{LABELS['rows'][r]} {v:.2f}" for r, v in sw.items())
                + f"; điểm số: {int((h4['mark'] == '+').sum())} ô đạo hàm đúng tốt hơn, "
                  f"{int((h4['mark'] == '−').sum())} ô kém hơn, {int((h4['mark'] == '=').sum())} ô trong 2·SE "
                  f"(trên {len(h4)} ô)"),
]
VERDICT = {"label": f"§4 {V[0]['verdict']}; §5 {V[1]['verdict']}; cầu nối {V[2]['verdict']}; H2 {V[3]['verdict']}; "
                    f"H1 {V[4]['verdict']}" + (" (CHẠY THỬ: chưa có giá trị kết luận)" if SMOKE else ""),
           "smoke": SMOKE, "n_images": N_IMAGES, "h2_counts": counts, "h2_bonferroni_worse": H2_BONF_WORSE, "rows": V}
(RUN_DIR / f"{NB_ID}_verdict.json").write_text(json.dumps(VERDICT, indent=2, ensure_ascii=False), encoding="utf-8")
lines = [f"# nb03 · kết luận ({N_IMAGES} ảnh test{', CHẠY THỬ: chưa có giá trị kết luận' if SMOKE else ''})", ""]
lines += [f"- **{v['hypothesis']}**: {v['verdict']}. {v['detail']}" for v in V]
(RUN_DIR / f"{NB_ID}_verdict.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("\n".join(lines))

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
display(Markdown(f"### Kết luận: **{json.loads((RUN_DIR / f'{NB_ID}_verdict.json').read_text(encoding='utf-8'))['label']}**"))
display(rep.verdict_table(RUN_DIR, LABELS))

## 15. Đổi nhãn và trình bày lại

Mọi cell hiển thị chỉ đọc `data/`, và lấy tên họ, tên hàng, tên cột, tiêu đề hình từ `LABELS`. Để đổi nhãn mà không tính lại:
1. nếu kernel vừa khởi động lại, chạy §1 (chỉ đặt đường dẫn, không tính gì);
2. sửa `LABELS` ở cell dưới rồi chạy nó;
3. chạy lại cell hiển thị cần thiết, hoặc cell cuối để vẽ lại tất cả. Hình được ghi đè vào `figures/`, bảng vào `tables/`.

Danh sách các khoá sửa được: `rep.default_labels()`. Tên phương pháp trong bảng ghép từ tên hàng (`LABELS["rows"]`) và tên họ (`LABELS["families"]`); LayerCAM và Random ở `LABELS["others"]`.

| File trong `data/` | Nội dung |
| --- | --- |
| `params.csv`, `rows.csv`, `env.json` | tham số, các hàng, môi trường (GPU, CPU, số worker) |
| `data_checks.csv`, `image_split.csv` | kiểm dữ liệu và baseline nb01; 566 ảnh test theo thứ tự của nb01, kèm lớp đích |
| `checks.csv`, `hm_fidelity.csv`, `float_precision.csv` | các phép kiểm của §4 |
| `scoring_check.csv` | kiểm đường chấm điểm với nb01 (§5) |
| `maps/image_KKK.npz` | bản đồ thô (có dấu, độ phân giải gốc của tầng) của 3 họ × 6 hàng, và đường tỉ số |
| `per_image/image_KKK.json`, `scores/image_KKK.csv` | kết quả theo ảnh của worker |
| `images.csv`, `alpha_star.csv`, `ratio_curves.npz`, `fig_images.npz` | ảnh, lớp đích; α*; đường tỉ số; ảnh minh hoạ |
| `bridge.csv`, `swap_effect.csv` | kiểm cầu nối từng ảnh; độ tương đồng giữa các họ (§11) |
| `scores_long.csv`, `all_long.csv`, `summary.csv` | điểm của nb03 dạng HM; điểm mọi phương pháp dạng dài (cả nb01); trung bình ± SE |
| `undefined_attrloc.csv` | số ảnh Attr. Loc. không xác định theo Quantus, theo phương pháp |
| `paired.csv`, `best_rows.csv` | hiệu ghép cặp (§9); hàng dẫn đầu và hàng tốt nhất |
| `timing.csv`, `run_log.csv`, `runtime_estimate.csv` | thời gian theo ảnh và giai đoạn; các lần chạy §6; thông lượng |

In [ ]:
importlib.reload(rep)
LABELS = rep.default_labels()
# ví dụ đổi nhãn:
# LABELS["families"]["gauge"] = "Gauge-X (balanced)"
# LABELS["rows"]["r3"] = "SplitCAM (26, +, g), wta"
# LABELS["metrics"]["pf_auc5"] = "Pixel Flipping AUC@5 ↑"

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
out = rep.redraw_all(RUN_DIR, LABELS)
print("đã vẽ lại:", ", ".join(out))